In [ ]:
# ============================================================
# STEP 1: Local Windows initialization
# Memory-safe circulant-graph experiment
# ============================================================

import os
import gc
import sys
import json
import random
import shutil
import platform
import subprocess
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import psutil


# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------

GLOBAL_SEED = 42

os.environ["PYTHONHASHSEED"] = str(GLOBAL_SEED)

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)


# ------------------------------------------------------------
# 2. Experiment directories
# ------------------------------------------------------------

EXPERIMENT_ROOT = Path(
    r"C:\Unicyclic_Bicyclic_Experiment_paper\Larger graph"
)

REPOSITORY_DIR = (
    EXPERIMENT_ROOT
    / "circulantGraphs"
)

DATA_DIR = (
    EXPERIMENT_ROOT
    / "data"
)

RAW_DATA_DIR = (
    DATA_DIR
    / "raw"
)

INTERIM_DATA_DIR = (
    DATA_DIR
    / "interim"
)

PROCESSED_DATA_DIR = (
    DATA_DIR
    / "processed"
)

CHECKPOINT_DIR = (
    EXPERIMENT_ROOT
    / "checkpoints"
)

RESULTS_DIR = (
    EXPERIMENT_ROOT
    / "results"
)

TABLES_DIR = (
    RESULTS_DIR
    / "tables"
)

MODELS_DIR = (
    RESULTS_DIR
    / "models"
)

LOGS_DIR = (
    RESULTS_DIR
    / "logs"
)

for directory in [
    EXPERIMENT_ROOT,
    DATA_DIR,
    RAW_DATA_DIR,
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    CHECKPOINT_DIR,
    RESULTS_DIR,
    TABLES_DIR,
    MODELS_DIR,
    LOGS_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# 3. Memory-monitoring utilities
# ------------------------------------------------------------

def get_memory_status():
    """
    Return current process and system RAM information.
    """

    process = psutil.Process(
        os.getpid()
    )

    virtual_memory = (
        psutil.virtual_memory()
    )

    return {
        "process_ram_gb": (
            process.memory_info().rss
            / (1024 ** 3)
        ),
        "system_ram_total_gb": (
            virtual_memory.total
            / (1024 ** 3)
        ),
        "system_ram_available_gb": (
            virtual_memory.available
            / (1024 ** 3)
        ),
        "system_ram_used_percent": (
            virtual_memory.percent
        ),
    }


def clear_temporary_memory():
    """
    Release unreachable Python objects without
    restarting or disconnecting the runtime.
    """

    gc.collect()


def print_memory_status(title="Memory status"):
    """
    Print current RAM utilization.
    """

    status = get_memory_status()

    print(f"\n{title}")
    print("-" * 70)

    for key, value in status.items():
        print(
            f"{key}: {value:.3f}"
        )


# ------------------------------------------------------------
# 4. Detect GPU without importing PyTorch yet
# ------------------------------------------------------------

def get_gpu_information():
    """
    Query the Colab GPU through nvidia-smi.
    """

    try:
        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu="
                "name,memory.total,memory.used",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            check=True,
        )

        return result.stdout.strip()

    except Exception:
        return "No NVIDIA GPU detected"


# ------------------------------------------------------------
# 5. Prepare the repository
# ------------------------------------------------------------

REPOSITORY_URL = (
    "https://github.com/"
    "RomeoMe5/circulantGraphs.git"
)

REPOSITORY_ZIP_URL = (
    "https://github.com/"
    "RomeoMe5/circulantGraphs/"
    "archive/refs/heads/master.zip"
)

# Reuse an existing valid repository instead of deleting it.
existing_csv_files = (
    list(REPOSITORY_DIR.rglob("*.csv"))
    if REPOSITORY_DIR.exists()
    else []
)

if existing_csv_files:
    print(
        "Existing repository detected; clone/download skipped:",
        REPOSITORY_DIR,
    )
else:
    if REPOSITORY_DIR.exists():
        shutil.rmtree(REPOSITORY_DIR)

    git_executable = shutil.which("git")

    if git_executable is not None:
        print("Git detected:", git_executable)
        print("Cloning repository...")

        subprocess.run(
            [
                git_executable,
                "clone",
                "--depth",
                "1",
                REPOSITORY_URL,
                str(REPOSITORY_DIR),
            ],
            check=True,
        )

    else:
        print(
            "Git was not found in PATH. "
            "Downloading the repository ZIP instead..."
        )

        repository_zip_path = (
            EXPERIMENT_ROOT
            / "circulantGraphs_master.zip"
        )

        extraction_directory = (
            EXPERIMENT_ROOT
            / "_repository_download"
        )

        if extraction_directory.exists():
            shutil.rmtree(extraction_directory)

        urllib.request.urlretrieve(
            REPOSITORY_ZIP_URL,
            repository_zip_path,
        )

        with zipfile.ZipFile(
            repository_zip_path,
            "r",
        ) as zip_handle:
            zip_handle.extractall(
                extraction_directory
            )

        extracted_candidates = [
            path
            for path in extraction_directory.iterdir()
            if path.is_dir()
        ]

        if len(extracted_candidates) != 1:
            raise RuntimeError(
                "Unable to identify the extracted repository directory."
            )

        shutil.move(
            str(extracted_candidates[0]),
            str(REPOSITORY_DIR),
        )

        repository_zip_path.unlink(
            missing_ok=True
        )

        shutil.rmtree(
            extraction_directory,
            ignore_errors=True,
        )

if not REPOSITORY_DIR.exists():
    raise FileNotFoundError(
        f"Repository directory was not created: {REPOSITORY_DIR}"
    )

# ------------------------------------------------------------
# 6. Count repository files without reading their contents
# ------------------------------------------------------------

repository_files = sum(
    1
    for path in REPOSITORY_DIR.rglob("*")
    if path.is_file()
)

csv_file_count = sum(
    1
    for path in REPOSITORY_DIR.rglob("*.csv")
    if path.is_file()
)


# ------------------------------------------------------------
# 7. Save experiment configuration
# ------------------------------------------------------------

experiment_configuration = {
    "global_seed": GLOBAL_SEED,
    "repository_url": REPOSITORY_URL,
    "repository_directory": str(
        REPOSITORY_DIR
    ),
    "experiment_root": str(
        EXPERIMENT_ROOT
    ),
    "target_graph_count": 200,
    "generator_dimensions": [
        2,
        3,
        4,
    ],
    "graphs_per_generator_dimension": {
        "2": 60,
        "3": 70,
        "4": 70,
    },
    "repeated_split_count": 10,
    "split_proportions": {
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15,
    },
    "memory_strategy": {
        "parallel_labeling": False,
        "store_all_graph_objects": False,
        "checkpoint_each_graph": True,
        "isolated_expensive_indices": True,
    },
}

CONFIGURATION_PATH = (
    EXPERIMENT_ROOT
    / "experiment_configuration.json"
)

with open(
    CONFIGURATION_PATH,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        experiment_configuration,
        file_handle,
        indent=2,
    )


# ------------------------------------------------------------
# 8. Environment summary
# ------------------------------------------------------------

print("=" * 72)
print("LOCAL CIRCULANT EXPERIMENT INITIALIZED")
print("=" * 72)

print(
    "Python version:",
    sys.version.split()[0],
)

print(
    "Platform:",
    platform.platform(),
)

print(
    "Experiment root:",
    EXPERIMENT_ROOT,
)

print(
    "Repository:",
    REPOSITORY_DIR,
)

print(
    "Repository files:",
    repository_files,
)

print(
    "CSV files:",
    csv_file_count,
)

print(
    "GPU:",
    get_gpu_information(),
)

print(
    "Configuration:",
    CONFIGURATION_PATH,
)

print_memory_status(
    "Initial RAM status"
)

assert REPOSITORY_DIR.exists()
assert repository_files > 0

print(
    "\nStep 1 completed successfully."
)

Git detected: /usr/bin/git
Cloning repository...
LOCAL CIRCULANT EXPERIMENT INITIALIZED
Python version: 3.12.13
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Experiment root: C:\Unicyclic_Bicyclic_Experiment_paper\Larger graph
Repository: C:\Unicyclic_Bicyclic_Experiment_paper\Larger graph/circulantGraphs
Repository files: 16667
CSV files: 16504
GPU: Tesla T4, 15360, 0
Configuration: C:\Unicyclic_Bicyclic_Experiment_paper\Larger graph/experiment_configuration.json

Initial RAM status
----------------------------------------------------------------------
process_ram_gb: 0.165
system_ram_total_gb: 12.671
system_ram_available_gb: 11.369
system_ram_used_percent: 10.300

Step 1 completed successfully.


In [ ]:
# ============================================================
# STEP 2A: Inspect repository CSV structure
# ============================================================

import csv
from pathlib import Path


# Build the repository CSV inventory before selecting samples.
repository_csv_files = sorted(
    path
    for path in REPOSITORY_DIR.rglob("*.csv")
    if path.is_file()
)

if not repository_csv_files:
    raise FileNotFoundError(
        f"No CSV files were found under: {REPOSITORY_DIR}"
    )

print(
    "Repository CSV files detected:",
    len(repository_csv_files),
)


sample_csv_files = repository_csv_files[:5]

print(
    "Sample files selected:",
    len(sample_csv_files),
)

for file_number, file_path in enumerate(
    sample_csv_files,
    start=1,
):

    print("\n" + "=" * 90)
    print(
        f"FILE {file_number}:",
        file_path.relative_to(
            REPOSITORY_DIR
        ),
    )
    print("=" * 90)

    raw_bytes = file_path.read_bytes()[:500]

    print(
        "First 100 raw bytes:"
    )
    print(
        repr(raw_bytes[:100])
    )

    successfully_read = False

    for encoding in [
        "utf-8-sig",
        "utf-8",
        "latin-1",
        "cp1252",
    ]:

        try:

            with open(
                file_path,
                "r",
                encoding=encoding,
                errors="strict",
            ) as file_handle:

                preview_lines = [
                    file_handle.readline()
                    for _ in range(12)
                ]

            print(
                "\nDetected readable encoding:",
                encoding,
            )

            print(
                "First non-empty lines:"
            )

            non_empty_counter = 0

            for line_index, line in enumerate(
                preview_lines,
                start=1,
            ):

                if line.strip():

                    print(
                        f"{line_index:02d}:",
                        repr(line.rstrip("\n\r")),
                    )

                    non_empty_counter += 1

                if non_empty_counter >= 8:
                    break

            combined_preview = "".join(
                preview_lines
            )

            try:

                dialect = csv.Sniffer().sniff(
                    combined_preview,
                    delimiters=[
                        ",",
                        ";",
                        "\t",
                        "|",
                    ],
                )

                print(
                    "Detected delimiter:",
                    repr(dialect.delimiter),
                )

            except csv.Error:

                print(
                    "Delimiter detection failed."
                )

            successfully_read = True
            break

        except UnicodeDecodeError:
            continue

    if not successfully_read:

        print(
            "Could not decode this file "
            "with the tested encodings."
        )


print_memory_status(
    "RAM status after file inspection"
)

print(
    "\nStep 2A completed."
)

Repository CSV files detected: 16504
Sample files selected: 5

FILE 1: Dataset/Dimension 02/C(N; D-1, D) circulant/formula_D_D-1_2-00003_2-00544.csv.csv
First 100 raw bytes:
b'\xd1\xe8\xe3\xed\xe0\xf2\xf3\xf0\xe0; \xc4\xe8\xe0\xec\xe5\xf2\xf0; \xd1\xf0\xe5\xe4\xed\xe5\xe5 \xf0\xe0\xf1\xf1\xf2\xee\xff\xed\xe8\xe5; \xc2\xf0\xe5\xec\xff \xe3\xe5\xed\xe5\xf0\xe0\xf6\xe8\xe8; \xca\xee\xeb-\xe2\xee \xf1\xee\xe5\xe4\xe8\xed\xe5\xed\xe8\xe9 \nSignature; Diameter; Ave'

Detected readable encoding: latin-1
First non-empty lines:
01: 'Ñèãíàòóðà; Äèàìåòð; Ñðåäíåå ðàññòîÿíèå; Âðåìÿ ãåíåðàöèè; Êîë-âî ñîåäèíåíèé '
02: 'Signature; Diameter; Average distance; Generation time; Number of connections '
03: '"C(3; 1, 2)";1;1;6;7'
04: '"C(4; 1, 2)";1;1;8;4'
05: '"C(5; 1, 2)";1;1;10;2'
06: '"C(6; 1, 2)";2;1,2;12;5'
07: '"C(7; 1, 2)";2;1,33333;14;6'
08: '"C(8; 1, 2)";2;1,42857;16;11'
Detected delimiter: ';'

FILE 2: Dataset/Dimension 02/optimal circulant1/2-00003_2-00550/2-00003.csv
First 100 raw bytes:
b'\xd

In [ ]:
# ============================================================
# STEP 2B: Correct memory-safe repository inventory
# ============================================================

import csv
import re
from collections import Counter


# ------------------------------------------------------------
# 1. Output paths
# ------------------------------------------------------------

INVENTORY_PATH = (
    INTERIM_DATA_DIR
    / "circulant_signature_inventory.csv"
)

INVENTORY_SUMMARY_PATH = (
    TABLES_DIR
    / "circulant_inventory_summary.csv"
)


# ------------------------------------------------------------
# 2. Parse circulant signatures
# ------------------------------------------------------------

def parse_circulant_signature(signature):
    """
    Parse a repository signature such as:

        C(20; 1, 3)
        C(20; 1, 3, 5)

    Returns
    -------
    tuple
        (order, canonical_generators)

    The repository's dimension refers to the number of
    listed generators in the signature.
    """

    if signature is None:
        return None

    signature = str(signature).strip()

    match = re.fullmatch(
        r'C\s*\(\s*(\d+)\s*;\s*([0-9,\s]+)\s*\)',
        signature,
        flags=re.IGNORECASE,
    )

    if match is None:
        return None

    order = int(
        match.group(1)
    )

    raw_generators = [
        int(value)
        for value in re.findall(
            r'\d+',
            match.group(2),
        )
    ]

    if not raw_generators:
        return None

    canonical_generators = tuple(
        sorted(
            set(
                min(
                    generator % order,
                    order - (
                        generator % order
                    ),
                )
                for generator
                in raw_generators
                if generator % order != 0
            )
        )
    )

    if not canonical_generators:
        return None

    return (
        order,
        canonical_generators,
    )


# ------------------------------------------------------------
# 3. Rebuild repository CSV inventory for this step
# ------------------------------------------------------------

repository_csv_files = sorted(
    path
    for path
    in REPOSITORY_DIR.rglob("*.csv")
    if path.is_file()
)

print(
    "Repository CSV files:",
    len(repository_csv_files),
)

assert len(repository_csv_files) > 0


# ------------------------------------------------------------
# 4. Stream inventory directly to disk
# ------------------------------------------------------------

inventory_columns = [
    "signature",
    "order",
    "generator_count",
    "generator_string",
    "source_file",
]

seen_signature_keys = set()

count_by_k = Counter()
orders_by_k = {}

files_processed = 0
files_skipped = 0
rows_examined = 0
rows_parsed = 0
invalid_rows = 0
duplicate_rows = 0

with open(
    INVENTORY_PATH,
    "w",
    encoding="utf-8",
    newline="",
) as inventory_handle:

    writer = csv.DictWriter(
        inventory_handle,
        fieldnames=inventory_columns,
    )

    writer.writeheader()

    for file_position, file_path in enumerate(
        repository_csv_files,
        start=1,
    ):

        try:

            with open(
                file_path,
                "r",
                encoding="latin-1",
                newline="",
            ) as source_handle:

                reader = csv.reader(
                    source_handle,
                    delimiter=";",
                    quotechar='"',
                    skipinitialspace=True,
                )

                # Row 1: Russian header
                next(
                    reader,
                    None,
                )

                # Row 2: English header
                english_header = next(
                    reader,
                    None,
                )

                if not english_header:

                    files_skipped += 1
                    continue

                normalized_header = [
                    str(value)
                    .strip()
                    .lower()
                    for value
                    in english_header
                ]

                if "signature" not in normalized_header:

                    files_skipped += 1
                    continue

                signature_column = (
                    normalized_header.index(
                        "signature"
                    )
                )

                for row in reader:

                    rows_examined += 1

                    if (
                        not row
                        or len(row)
                        <= signature_column
                    ):
                        invalid_rows += 1
                        continue

                    original_signature = (
                        row[
                            signature_column
                        ].strip()
                    )

                    parsed = (
                        parse_circulant_signature(
                            original_signature
                        )
                    )

                    if parsed is None:

                        invalid_rows += 1
                        continue

                    order, generators = parsed

                    generator_count = len(
                        generators
                    )

                    signature_key = (
                        order,
                        generators,
                    )

                    if (
                        signature_key
                        in seen_signature_keys
                    ):

                        duplicate_rows += 1
                        continue

                    seen_signature_keys.add(
                        signature_key
                    )

                    generator_string = ",".join(
                        str(generator)
                        for generator
                        in generators
                    )

                    canonical_signature = (
                        f"C({order}; "
                        f"{generator_string})"
                    )

                    writer.writerow({
                        "signature": (
                            canonical_signature
                        ),
                        "order": int(order),
                        "generator_count": int(
                            generator_count
                        ),
                        "generator_string": (
                            generator_string
                        ),
                        "source_file": str(
                            file_path.relative_to(
                                REPOSITORY_DIR
                            )
                        ),
                    })

                    rows_parsed += 1

                    count_by_k[
                        generator_count
                    ] += 1

                    orders_by_k.setdefault(
                        generator_count,
                        set(),
                    ).add(
                        order
                    )

            files_processed += 1

        except (
            UnicodeDecodeError,
            csv.Error,
            OSError,
        ) as exception:

            files_skipped += 1

        if (
            file_position % 1000 == 0
            or file_position
            == len(repository_csv_files)
        ):

            print(
                f"Scanned "
                f"{file_position:,}/"
                f"{len(repository_csv_files):,} files | "
                f"unique signatures: "
                f"{rows_parsed:,}",
                flush=True,
            )

            gc.collect()


# ------------------------------------------------------------
# 5. Compact summary table
# ------------------------------------------------------------

summary_rows = []

for generator_count in sorted(
    count_by_k.keys()
):

    available_orders = sorted(
        orders_by_k.get(
            generator_count,
            set(),
        )
    )

    summary_rows.append({
        "generator_count": int(
            generator_count
        ),
        "graph_signatures": int(
            count_by_k[
                generator_count
            ]
        ),
        "minimum_order": int(
            min(available_orders)
        ),
        "maximum_order": int(
            max(available_orders)
        ),
        "unique_orders": int(
            len(available_orders)
        ),
    })

df_inventory_summary = pd.DataFrame(
    summary_rows
)

df_inventory_summary.to_csv(
    INVENTORY_SUMMARY_PATH,
    index=False,
)


# ------------------------------------------------------------
# 6. Validate target dimensions
# ------------------------------------------------------------

TARGET_K_VALUES = [
    2,
    3,
    4,
]

missing_target_k = [
    k_value
    for k_value in TARGET_K_VALUES
    if count_by_k.get(
        k_value,
        0,
    ) == 0
]


# ------------------------------------------------------------
# 7. Report
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("CORRECTED REPOSITORY INVENTORY COMPLETE")
print("=" * 72)

print(
    "Files processed:",
    f"{files_processed:,}",
)

print(
    "Files skipped:",
    f"{files_skipped:,}",
)

print(
    "Rows examined:",
    f"{rows_examined:,}",
)

print(
    "Unique parsed signatures:",
    f"{rows_parsed:,}",
)

print(
    "Duplicate signatures removed:",
    f"{duplicate_rows:,}",
)

print(
    "Invalid rows:",
    f"{invalid_rows:,}",
)

print(
    "\nInventory summary:"
)

print(
    df_inventory_summary.to_string(
        index=False
    )
)

print(
    "\nTarget dimensions available:",
    len(missing_target_k) == 0,
)

if missing_target_k:

    print(
        "Missing target dimensions:",
        missing_target_k,
    )

print(
    "\nInventory saved to:"
)

print(
    INVENTORY_PATH
)

print(
    "Summary saved to:"
)

print(
    INVENTORY_SUMMARY_PATH
)

print_memory_status(
    "RAM status after corrected inventory"
)

assert rows_parsed > 0

if missing_target_k:
    raise RuntimeError(
        "Required generator dimensions are missing: "
        f"{missing_target_k}"
    )

print(
    "\nStep 2B completed successfully."
)

Repository CSV files: 16504
Scanned 1,000/16,504 files | unique signatures: 24,558
Scanned 2,000/16,504 files | unique signatures: 50,316
Scanned 14,000/16,504 files | unique signatures: 61,142

CORRECTED REPOSITORY INVENTORY COMPLETE
Files processed: 2,642
Files skipped: 13,862
Rows examined: 379,378
Unique parsed signatures: 61,142
Duplicate signatures removed: 2,130
Invalid rows: 316,106

Inventory summary:
 generator_count  graph_signatures  minimum_order  maximum_order  unique_orders
               1                 1              3              3              1
               2             24921              4            550            547
               3             27688              6            529            504
               4              8532              8            484            157

Target dimensions available: True

Inventory saved to:
C:\Unicyclic_Bicyclic_Experiment_paper\Larger graph/data/interim/circulant_signature_inventory.csv
Summary saved to:
C:\Unicyclic_

In [ ]:
# ============================================================
# STEP 3: Inspect candidate availability by graph order
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# 1. Load only the compact inventory columns required here
# ------------------------------------------------------------

df_inventory_counts = pd.read_csv(
    INVENTORY_PATH,
    usecols=[
        "order",
        "generator_count",
    ],
    dtype={
        "order": "int32",
        "generator_count": "int8",
    },
)

TARGET_K_VALUES = [2, 3, 4]

df_inventory_counts = (
    df_inventory_counts.loc[
        df_inventory_counts[
            "generator_count"
        ].isin(TARGET_K_VALUES)
    ]
    .copy()
)


# ------------------------------------------------------------
# 2. Count available signatures for each order and k
# ------------------------------------------------------------

df_order_availability = (
    df_inventory_counts
    .groupby(
        [
            "order",
            "generator_count",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "available_signatures"
        }
    )
)

df_order_pivot = (
    df_order_availability
    .pivot(
        index="order",
        columns="generator_count",
        values="available_signatures",
    )
    .fillna(0)
    .astype(int)
    .rename(
        columns={
            2: "k2",
            3: "k3",
            4: "k4",
        }
    )
    .reset_index()
)

for required_column in [
    "k2",
    "k3",
    "k4",
]:
    if required_column not in df_order_pivot:
        df_order_pivot[
            required_column
        ] = 0

df_order_pivot[
    "total"
] = (
    df_order_pivot[
        ["k2", "k3", "k4"]
    ].sum(axis=1)
)


# ------------------------------------------------------------
# 3. Restrict display to computationally relevant orders
# ------------------------------------------------------------

INSPECTION_MIN_ORDER = 16
INSPECTION_MAX_ORDER = 36

df_feasible_order_window = (
    df_order_pivot.loc[
        df_order_pivot[
            "order"
        ].between(
            INSPECTION_MIN_ORDER,
            INSPECTION_MAX_ORDER,
        )
    ]
    .sort_values("order")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. Summaries for possible benchmark windows
# ------------------------------------------------------------

candidate_windows = [
    (16, 24),
    (18, 26),
    (20, 28),
    (20, 30),
    (20, 32),
    (22, 32),
]

window_summary_rows = []

for minimum_order, maximum_order in candidate_windows:

    current_window = (
        df_order_pivot.loc[
            df_order_pivot[
                "order"
            ].between(
                minimum_order,
                maximum_order,
            )
        ]
    )

    window_summary_rows.append({
        "order_window": (
            f"{minimum_order}--{maximum_order}"
        ),
        "k2_available": int(
            current_window["k2"].sum()
        ),
        "k3_available": int(
            current_window["k3"].sum()
        ),
        "k4_available": int(
            current_window["k4"].sum()
        ),
        "total_available": int(
            current_window["total"].sum()
        ),
        "unique_orders": int(
            current_window["order"].nunique()
        ),
    })

df_candidate_window_summary = pd.DataFrame(
    window_summary_rows
)

ORDER_AVAILABILITY_PATH = (
    TABLES_DIR
    / "circulant_order_availability.csv"
)

WINDOW_SUMMARY_PATH = (
    TABLES_DIR
    / "circulant_candidate_window_summary.csv"
)

df_order_pivot.to_csv(
    ORDER_AVAILABILITY_PATH,
    index=False,
)

df_candidate_window_summary.to_csv(
    WINDOW_SUMMARY_PATH,
    index=False,
)


# ------------------------------------------------------------
# 5. Output
# ------------------------------------------------------------

print("=" * 78)
print("CANDIDATE AVAILABILITY BY ORDER")
print("=" * 78)

print(
    df_feasible_order_window.to_string(
        index=False
    )
)

print("\nCandidate-window summary")
print("-" * 78)

print(
    df_candidate_window_summary.to_string(
        index=False
    )
)

print(
    "\nTotal repository signatures retained "
    "for k=2,3,4:",
    f"{len(df_inventory_counts):,}",
)

print(
    "\nOrder availability saved to:"
)

print(
    ORDER_AVAILABILITY_PATH
)

print(
    "Window summary saved to:"
)

print(
    WINDOW_SUMMARY_PATH
)

print_memory_status(
    "RAM status after availability analysis"
)

del df_inventory_counts
clear_temporary_memory()

print(
    "\nStep 3 completed successfully."
)

CANDIDATE AVAILABILITY BY ORDER
 order  k2  k3  k4  total
    16   4  10  13     27
    17  10  13  17     40
    18   8   7  13     28
    19  13  11  20     44
    20   4   7  18     29
    21   6  10  23     39
    22   5   8  17     30
    23   9  15  24     48
    24   2   2  21     25
    25   5   9  26     40
    26   5  10  19     34
    27   7   9  28     44
    28   9   5  22     36
    29  17  10  28     55
    30   3   9  10     22
    31  21  16  28     65
    32  15  15  10     40
    33  20  22  23     65
    34   7  23  10     40
    35  11  28  10     49
    36   9  27  21     57

Candidate-window summary
------------------------------------------------------------------------------
order_window  k2_available  k3_available  k4_available  total_available  unique_orders
      16--24            61            83           166              310              9
      18--26            57            79           181              317              9
      20--28            52    

In [ ]:
# ============================================================
# STEP 4: Select primary benchmark and reserve candidates
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Selection configuration
# ------------------------------------------------------------

SELECTION_SEED = 42

MIN_ORDER = 16
MAX_ORDER = 26

DEVELOPMENT_MAX_ORDER = 23
HELDOUT_MIN_ORDER = 24

# Final 200-graph composition
FINAL_COUNTS_BY_K = {
    2: 60,
    3: 70,
    4: 70,
}

# Independent higher-order test composition
HELDOUT_COUNTS_BY_K = {
    2: 10,
    3: 15,
    4: 20,
}

assert sum(FINAL_COUNTS_BY_K.values()) == 200
assert sum(HELDOUT_COUNTS_BY_K.values()) == 45


# ------------------------------------------------------------
# 2. Load candidate metadata only
# ------------------------------------------------------------

inventory_columns = [
    "signature",
    "order",
    "generator_count",
    "generator_string",
    "source_file",
]

df_candidates = pd.read_csv(
    INVENTORY_PATH,
    usecols=inventory_columns,
    dtype={
        "signature": "string",
        "order": "int32",
        "generator_count": "int8",
        "generator_string": "string",
        "source_file": "string",
    },
)

df_candidates = (
    df_candidates.loc[
        df_candidates["order"].between(
            MIN_ORDER,
            MAX_ORDER,
        )
        &
        df_candidates["generator_count"].isin(
            [2, 3, 4]
        )
    ]
    .copy()
)

df_candidates["order_group"] = np.where(
    df_candidates["order"]
    <= DEVELOPMENT_MAX_ORDER,
    "development",
    "heldout",
)


# ------------------------------------------------------------
# 3. Check available counts
# ------------------------------------------------------------

availability = (
    df_candidates
    .groupby(
        [
            "generator_count",
            "order_group",
        ]
    )
    .size()
    .unstack(fill_value=0)
)

print("=" * 76)
print("AVAILABLE CANDIDATES")
print("=" * 76)
print(availability)


# ------------------------------------------------------------
# 4. Balanced sampling across graph orders
# ------------------------------------------------------------

def balanced_sample_by_order(
    dataframe,
    required_count,
    random_seed,
):
    """
    Select graphs approximately uniformly across available
    orders without constructing graph objects.
    """

    dataframe = dataframe.copy()

    available_orders = sorted(
        dataframe["order"].unique()
    )

    if len(dataframe) < required_count:
        raise ValueError(
            f"Requested {required_count} graphs, "
            f"but only {len(dataframe)} are available."
        )

    rng = np.random.default_rng(
        random_seed
    )

    shuffled_groups = {}

    for order in available_orders:

        order_rows = dataframe.loc[
            dataframe["order"] == order
        ]

        shuffled_indices = rng.permutation(
            order_rows.index.to_numpy()
        )

        shuffled_groups[order] = list(
            shuffled_indices
        )

    selected_indices = []

    # Round-robin allocation across orders
    while (
        len(selected_indices) < required_count
    ):

        added_this_round = False

        for order in available_orders:

            if shuffled_groups[order]:

                selected_indices.append(
                    shuffled_groups[order].pop()
                )

                added_this_round = True

                if (
                    len(selected_indices)
                    == required_count
                ):
                    break

        if not added_this_round:
            break

    return dataframe.loc[
        selected_indices
    ].copy()


# ------------------------------------------------------------
# 5. Select the primary 200 graphs
# ------------------------------------------------------------

primary_parts = []

for k_value in [2, 3, 4]:

    total_required = (
        FINAL_COUNTS_BY_K[k_value]
    )

    heldout_required = (
        HELDOUT_COUNTS_BY_K[k_value]
    )

    development_required = (
        total_required
        - heldout_required
    )

    development_pool = df_candidates.loc[
        (
            df_candidates["generator_count"]
            == k_value
        )
        &
        (
            df_candidates["order_group"]
            == "development"
        )
    ]

    heldout_pool = df_candidates.loc[
        (
            df_candidates["generator_count"]
            == k_value
        )
        &
        (
            df_candidates["order_group"]
            == "heldout"
        )
    ]

    selected_development = (
        balanced_sample_by_order(
            dataframe=development_pool,
            required_count=development_required,
            random_seed=(
                SELECTION_SEED
                + 100 * k_value
            ),
        )
    )

    selected_heldout = (
        balanced_sample_by_order(
            dataframe=heldout_pool,
            required_count=heldout_required,
            random_seed=(
                SELECTION_SEED
                + 1000
                + 100 * k_value
            ),
        )
    )

    primary_parts.extend([
        selected_development,
        selected_heldout,
    ])


df_primary = pd.concat(
    primary_parts,
    ignore_index=True,
)

df_primary = (
    df_primary
    .sort_values(
        [
            "order_group",
            "generator_count",
            "order",
            "signature",
        ]
    )
    .reset_index(drop=True)
)

df_primary.insert(
    0,
    "graph_id",
    np.arange(
        len(df_primary),
        dtype=np.int32,
    ),
)

df_primary["selection_status"] = (
    "primary"
)


# ------------------------------------------------------------
# 6. Create reserve pool
# ------------------------------------------------------------

selected_signatures = set(
    df_primary["signature"].astype(str)
)

df_reserve = (
    df_candidates.loc[
        ~df_candidates["signature"]
        .astype(str)
        .isin(selected_signatures)
    ]
    .copy()
)

# Reproducible reserve ordering within each stratum
df_reserve = (
    df_reserve
    .sample(
        frac=1.0,
        random_state=SELECTION_SEED,
    )
    .sort_values(
        [
            "order_group",
            "generator_count",
            "order",
        ]
    )
    .reset_index(drop=True)
)

df_reserve.insert(
    0,
    "reserve_id",
    np.arange(
        len(df_reserve),
        dtype=np.int32,
    ),
)

df_reserve["selection_status"] = (
    "reserve"
)


# ------------------------------------------------------------
# 7. Save manifests
# ------------------------------------------------------------

PRIMARY_MANIFEST_PATH = (
    INTERIM_DATA_DIR
    / "circulant_primary_200.csv"
)

RESERVE_MANIFEST_PATH = (
    INTERIM_DATA_DIR
    / "circulant_reserve_pool.csv"
)

df_primary.to_csv(
    PRIMARY_MANIFEST_PATH,
    index=False,
)

df_reserve.to_csv(
    RESERVE_MANIFEST_PATH,
    index=False,
)


# ------------------------------------------------------------
# 8. Generate benchmark-composition table
# ------------------------------------------------------------

composition_rows = []

for k_value in [2, 3, 4]:

    current = df_primary.loc[
        df_primary["generator_count"]
        == k_value
    ]

    composition_rows.append({
        "generator_dimension": (
            f"k={k_value}"
        ),
        "graphs": int(
            len(current)
        ),
        "minimum_order": int(
            current["order"].min()
        ),
        "maximum_order": int(
            current["order"].max()
        ),
        "unique_orders": int(
            current["order"].nunique()
        ),
        "development_graphs": int(
            (
                current["order_group"]
                == "development"
            ).sum()
        ),
        "heldout_graphs": int(
            (
                current["order_group"]
                == "heldout"
            ).sum()
        ),
    })

composition_rows.append({
    "generator_dimension": "Total",
    "graphs": int(len(df_primary)),
    "minimum_order": int(
        df_primary["order"].min()
    ),
    "maximum_order": int(
        df_primary["order"].max()
    ),
    "unique_orders": int(
        df_primary["order"].nunique()
    ),
    "development_graphs": int(
        (
            df_primary["order_group"]
            == "development"
        ).sum()
    ),
    "heldout_graphs": int(
        (
            df_primary["order_group"]
            == "heldout"
        ).sum()
    ),
})

df_benchmark_composition = pd.DataFrame(
    composition_rows
)

BENCHMARK_COMPOSITION_PATH = (
    TABLES_DIR
    / "circulant_benchmark_composition.csv"
)

df_benchmark_composition.to_csv(
    BENCHMARK_COMPOSITION_PATH,
    index=False,
)


# ------------------------------------------------------------
# 9. Validation
# ------------------------------------------------------------

assert len(df_primary) == 200
assert df_primary["signature"].nunique() == 200

assert (
    df_primary.loc[
        df_primary["order_group"]
        == "heldout"
    ].shape[0]
    == 45
)

for k_value, expected_count in (
    FINAL_COUNTS_BY_K.items()
):

    observed_count = int(
        (
            df_primary["generator_count"]
            == k_value
        ).sum()
    )

    assert observed_count == expected_count

for k_value, expected_count in (
    HELDOUT_COUNTS_BY_K.items()
):

    observed_count = int(
        (
            (
                df_primary["generator_count"]
                == k_value
            )
            &
            (
                df_primary["order_group"]
                == "heldout"
            )
        ).sum()
    )

    assert observed_count == expected_count


# ------------------------------------------------------------
# 10. Output
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("PRIMARY BENCHMARK COMPOSITION")
print("=" * 76)

print(
    df_benchmark_composition.to_string(
        index=False
    )
)

print("\nGraphs by exact order and generator dimension")
print("-" * 76)

order_distribution = (
    df_primary
    .pivot_table(
        index="order",
        columns="generator_count",
        values="signature",
        aggfunc="count",
        fill_value=0,
    )
    .rename(
        columns={
            2: "k2",
            3: "k3",
            4: "k4",
        }
    )
)

order_distribution["total"] = (
    order_distribution.sum(axis=1)
)

print(
    order_distribution.to_string()
)

print(
    "\nPrimary graphs:",
    len(df_primary),
)

print(
    "Development graphs:",
    int(
        (
            df_primary["order_group"]
            == "development"
        ).sum()
    ),
)

print(
    "Held-out graphs:",
    int(
        (
            df_primary["order_group"]
            == "heldout"
        ).sum()
    ),
)

print(
    "Reserve candidates:",
    len(df_reserve),
)

print(
    "\nPrimary manifest:"
)

print(
    PRIMARY_MANIFEST_PATH
)

print(
    "Reserve manifest:"
)

print(
    RESERVE_MANIFEST_PATH
)

print(
    "Composition table:"
)

print(
    BENCHMARK_COMPOSITION_PATH
)

print_memory_status(
    "RAM status after candidate selection"
)

del df_candidates
clear_temporary_memory()

print(
    "\nStep 4 completed successfully."
)

AVAILABLE CANDIDATES
order_group      development  heldout
generator_count                      
2                         59       12
3                         81       21
4                        145       66

PRIMARY BENCHMARK COMPOSITION
generator_dimension  graphs  minimum_order  maximum_order  unique_orders  development_graphs  heldout_graphs
                k=2      60             16             26             11                  50              10
                k=3      70             16             26             11                  55              15
                k=4      70             16             26             11                  50              20
              Total     200             16             26             11                 155              45

Graphs by exact order and generator dimension
----------------------------------------------------------------------------
generator_count  k2  k3  k4  total
order                             
16                4

In [ ]:
# ============================================================
# STEP 5: Validate graph reconstruction and fast indices
# ============================================================

import math
import ast
import networkx as nx
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Load a small representative sample
# ------------------------------------------------------------

df_primary_check = pd.read_csv(
    PRIMARY_MANIFEST_PATH,
    dtype={
        "graph_id": "int32",
        "signature": "string",
        "order": "int32",
        "generator_count": "int8",
        "generator_string": "string",
        "order_group": "string",
    },
)

validation_rows = []

for k_value in [2, 3, 4]:

    current = df_primary_check.loc[
        df_primary_check["generator_count"] == k_value
    ]

    validation_rows.append(
        current.iloc[0]
    )

    validation_rows.append(
        current.iloc[-1]
    )

df_validation_sample = pd.DataFrame(
    validation_rows
).reset_index(drop=True)

print(
    "Validation sample size:",
    len(df_validation_sample),
)


# ------------------------------------------------------------
# 2. Parse generator strings
# ------------------------------------------------------------

def parse_generator_string(
    generator_string,
):
    """
    Convert a stored generator string such as '1,4,7'
    into a tuple of integers.
    """

    values = [
        int(value.strip())
        for value in str(
            generator_string
        ).split(",")
        if value.strip()
    ]

    return tuple(values)


# ------------------------------------------------------------
# 3. Reconstruct one undirected circulant graph
# ------------------------------------------------------------

def build_circulant_graph(
    order,
    generators,
):
    """
    Construct the undirected circulant graph C(n; s1,...,sk).

    For each node v and generator s, edges are added between
    v and (v+s) mod n. NetworkX automatically removes duplicate
    edges in the undirected graph.
    """

    order = int(order)

    generators = tuple(
        int(generator)
        for generator in generators
    )

    graph = nx.Graph()

    graph.add_nodes_from(
        range(order)
    )

    for node in range(order):

        for generator in generators:

            neighbour = (
                node + generator
            ) % order

            if neighbour != node:

                graph.add_edge(
                    node,
                    neighbour,
                )

    return graph


# ------------------------------------------------------------
# 4. Fast exact topological indices
# ------------------------------------------------------------

def compute_wiener_index(
    graph,
):
    """
    Wiener index:
        W(G) = sum_{u<v} d(u,v)
    """

    return int(
        nx.wiener_index(graph)
    )


def compute_first_zagreb_index(
    graph,
):
    """
    First Zagreb index:
        M1(G) = sum_v d(v)^2
    """

    return int(
        sum(
            degree ** 2
            for _, degree
            in graph.degree()
        )
    )


def compute_second_zagreb_index(
    graph,
):
    """
    Second Zagreb index:
        M2(G) = sum_{uv in E} d(u)d(v)
    """

    degrees = dict(
        graph.degree()
    )

    return int(
        sum(
            degrees[source]
            * degrees[target]
            for source, target
            in graph.edges()
        )
    )


def compute_randic_index(
    graph,
):
    """
    Randic index:
        R(G) = sum_{uv in E} 1/sqrt(d(u)d(v))
    """

    degrees = dict(
        graph.degree()
    )

    return float(
        sum(
            1.0
            / math.sqrt(
                degrees[source]
                * degrees[target]
            )
            for source, target
            in graph.edges()
        )
    )


# ------------------------------------------------------------
# 5. Validate representative graphs
# ------------------------------------------------------------

validation_results = []

for _, row in df_validation_sample.iterrows():

    graph_id = int(
        row["graph_id"]
    )

    order = int(
        row["order"]
    )

    generators = (
        parse_generator_string(
            row["generator_string"]
        )
    )

    graph = build_circulant_graph(
        order=order,
        generators=generators,
    )

    connected = nx.is_connected(
        graph
    )

    node_count = graph.number_of_nodes()
    edge_count = graph.number_of_edges()

    degree_values = [
        degree
        for _, degree
        in graph.degree()
    ]

    regular = (
        len(set(degree_values))
        == 1
    )

    regular_degree = (
        degree_values[0]
        if regular
        else None
    )

    W = compute_wiener_index(
        graph
    )

    M1 = compute_first_zagreb_index(
        graph
    )

    M2 = compute_second_zagreb_index(
        graph
    )

    R = compute_randic_index(
        graph
    )

    validation_results.append({
        "graph_id": graph_id,
        "signature": str(
            row["signature"]
        ),
        "k": int(
            row["generator_count"]
        ),
        "order": order,
        "generators": str(
            generators
        ),
        "nodes": node_count,
        "edges": edge_count,
        "connected": connected,
        "regular": regular,
        "regular_degree": (
            regular_degree
        ),
        "W": W,
        "M1": M1,
        "M2": M2,
        "R": R,
    })

    del graph
    gc.collect()


df_validation_results = pd.DataFrame(
    validation_results
)


# ------------------------------------------------------------
# 6. Structural consistency checks
# ------------------------------------------------------------

assert (
    df_validation_results["nodes"]
    == df_validation_results["order"]
).all()

assert (
    df_validation_results["connected"]
).all()

assert (
    df_validation_results["regular"]
).all()

assert (
    df_validation_results["W"]
    > 0
).all()

assert (
    df_validation_results["M1"]
    > 0
).all()

assert (
    df_validation_results["M2"]
    > 0
).all()

assert (
    df_validation_results["R"]
    > 0
).all()


# ------------------------------------------------------------
# 7. Save validation output
# ------------------------------------------------------------

VALIDATION_RESULTS_PATH = (
    TABLES_DIR
    / "circulant_reconstruction_validation.csv"
)

df_validation_results.to_csv(
    VALIDATION_RESULTS_PATH,
    index=False,
)


# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CIRCULANT RECONSTRUCTION VALIDATION")
print("=" * 100)

print(
    df_validation_results.to_string(
        index=False
    )
)

print(
    "\nValidation file:"
)

print(
    VALIDATION_RESULTS_PATH
)

print_memory_status(
    "RAM status after reconstruction validation"
)

del df_primary_check
del df_validation_sample
clear_temporary_memory()

print(
    "\nStep 5 completed successfully."
)

Validation sample size: 6

CIRCULANT RECONSTRUCTION VALIDATION
 graph_id        signature  k  order     generators  nodes  edges  connected  regular  regular_degree   W   M1   M2    R
        0       C(16; 1,6)  2     16         (1, 6)     16     32       True     True               4 232  256  512  8.0
      164      C(26; 9,12)  2     26        (9, 12)     26     52       True     True               4 780  416  832 13.0
       50     C(16; 1,3,4)  3     16      (1, 3, 4)     16     48       True     True               6 192  576 1728  8.0
      179   C(26; 9,10,12)  3     26    (9, 10, 12)     26     78       True     True               6 611  936 2808 13.0
      105   C(16; 1,2,3,5)  4     16   (1, 2, 3, 5)     16     64       True     True               8 176 1024 4096  8.0
      199 C(26; 5,8,11,12)  4     26 (5, 8, 11, 12)     26    104       True     True               8 546 1664 6656 13.0

Validation file:
C:\Unicyclic_Bicyclic_Experiment_paper\Larger graph/results/tables/circu

In [ ]:
# ============================================================
# STEP 6: Memory-safe pilot for exact MS(G) and Z(G)
# ============================================================

import os
import gc
import time
import queue
import multiprocessing as mp
from functools import lru_cache

import pandas as pd
import psutil


# ------------------------------------------------------------
# 1. Pilot configuration
# ------------------------------------------------------------

PILOT_TIMEOUT_SECONDS = 90

PILOT_RESULTS_PATH = (
    TABLES_DIR
    / "circulant_exact_labeling_pilot.csv"
)


# ------------------------------------------------------------
# 2. Convert a circulant graph to adjacency bitmasks
# ------------------------------------------------------------

def build_adjacency_masks(order, generators):
    """
    Construct adjacency bitmasks without retaining a NetworkX graph.

    adjacency_masks[v] contains a bit for every neighbor of v.
    """

    order = int(order)
    generators = tuple(
        int(generator)
        for generator in generators
    )

    adjacency_masks = [0] * order

    for vertex in range(order):

        for generator in generators:

            forward = (
                vertex + generator
            ) % order

            backward = (
                vertex - generator
            ) % order

            if forward != vertex:
                adjacency_masks[vertex] |= (
                    1 << forward
                )

            if backward != vertex:
                adjacency_masks[vertex] |= (
                    1 << backward
                )

    return tuple(adjacency_masks)


# ------------------------------------------------------------
# 3. Vertex selection heuristic
# ------------------------------------------------------------

def select_active_vertex(
    active_mask,
    adjacency_masks,
):
    """
    Select an active vertex having the largest degree in the
    current induced subgraph.

    This reduces recursive branching compared with always
    selecting the first available vertex.
    """

    best_vertex = -1
    best_degree = -1

    remaining = active_mask

    while remaining:

        lowest_bit = (
            remaining
            & -remaining
        )

        vertex = (
            lowest_bit.bit_length()
            - 1
        )

        degree = (
            adjacency_masks[vertex]
            & active_mask
        ).bit_count()

        if degree > best_degree:

            best_vertex = vertex
            best_degree = degree

        remaining ^= lowest_bit

    return best_vertex


# ------------------------------------------------------------
# 4. Exact Merrifield--Simmons index
# ------------------------------------------------------------

def exact_merrifield_simmons(
    adjacency_masks,
):
    """
    Count all independent sets.

    Recurrence:
        I(G) = I(G-v) + I(G-v-N(v))
    """

    order = len(adjacency_masks)
    full_mask = (
        1 << order
    ) - 1

    @lru_cache(maxsize=None)
    def count_independent_sets(
        active_mask,
    ):

        if active_mask == 0:
            return 1

        vertex = select_active_vertex(
            active_mask,
            adjacency_masks,
        )

        vertex_bit = (
            1 << vertex
        )

        # Exclude v
        without_vertex = (
            active_mask
            & ~vertex_bit
        )

        # Include v: remove v and all active neighbors
        without_closed_neighborhood = (
            active_mask
            & ~vertex_bit
            & ~adjacency_masks[vertex]
        )

        return (
            count_independent_sets(
                without_vertex
            )
            +
            count_independent_sets(
                without_closed_neighborhood
            )
        )

    value = count_independent_sets(
        full_mask
    )

    cache_info = (
        count_independent_sets.cache_info()
    )

    return int(value), int(
        cache_info.currsize
    )


# ------------------------------------------------------------
# 5. Exact Hosoya index
# ------------------------------------------------------------

def exact_hosoya_index(
    adjacency_masks,
):
    """
    Count all matchings, including the empty matching.

    For an active vertex v:

        Z(G) = Z(G-v)
               + sum_{u in N(v)} Z(G-v-u)

    The first term leaves v unmatched. Each remaining term
    matches v with exactly one active neighbor u.
    """

    order = len(adjacency_masks)
    full_mask = (
        1 << order
    ) - 1

    @lru_cache(maxsize=None)
    def count_matchings(
        active_mask,
    ):

        if active_mask == 0:
            return 1

        vertex = select_active_vertex(
            active_mask,
            adjacency_masks,
        )

        vertex_bit = (
            1 << vertex
        )

        without_vertex = (
            active_mask
            & ~vertex_bit
        )

        # Case 1: v remains unmatched
        total = count_matchings(
            without_vertex
        )

        # Case 2: v is matched with one active neighbor
        active_neighbors = (
            adjacency_masks[vertex]
            & without_vertex
        )

        while active_neighbors:

            neighbor_bit = (
                active_neighbors
                & -active_neighbors
            )

            total += count_matchings(
                without_vertex
                & ~neighbor_bit
            )

            active_neighbors ^= (
                neighbor_bit
            )

        return total

    value = count_matchings(
        full_mask
    )

    cache_info = (
        count_matchings.cache_info()
    )

    return int(value), int(
        cache_info.currsize
    )


# ------------------------------------------------------------
# 6. Child-process worker
# ------------------------------------------------------------

def exact_index_worker(
    index_name,
    adjacency_masks,
    result_queue,
):
    """
    Execute one exact index inside an isolated process.
    """

    try:

        start_time = time.perf_counter()

        process = psutil.Process(
            os.getpid()
        )

        initial_memory_mb = (
            process.memory_info().rss
            / (1024 ** 2)
        )

        if index_name == "MS":

            value, cache_states = (
                exact_merrifield_simmons(
                    adjacency_masks
                )
            )

        elif index_name == "Z":

            value, cache_states = (
                exact_hosoya_index(
                    adjacency_masks
                )
            )

        else:

            raise ValueError(
                f"Unsupported index: {index_name}"
            )

        elapsed_seconds = (
            time.perf_counter()
            - start_time
        )

        peak_observed_memory_mb = (
            process.memory_info().rss
            / (1024 ** 2)
        )

        result_queue.put({
            "status": "completed",
            "value": value,
            "elapsed_seconds": (
                elapsed_seconds
            ),
            "cache_states": (
                cache_states
            ),
            "initial_memory_mb": (
                initial_memory_mb
            ),
            "final_memory_mb": (
                peak_observed_memory_mb
            ),
            "error": None,
        })

    except Exception as exception:

        result_queue.put({
            "status": "error",
            "value": None,
            "elapsed_seconds": None,
            "cache_states": None,
            "initial_memory_mb": None,
            "final_memory_mb": None,
            "error": repr(exception),
        })


# ------------------------------------------------------------
# 7. Timeout-controlled isolated execution
# ------------------------------------------------------------

def run_exact_index_isolated(
    index_name,
    adjacency_masks,
    timeout_seconds,
):
    """
    Run one exact index in a separate process.

    Only the child process is terminated on timeout.
    The Colab notebook process remains active.
    """

    context = mp.get_context(
        "fork"
    )

    result_queue = context.Queue(
        maxsize=1
    )

    process = context.Process(
        target=exact_index_worker,
        args=(
            index_name,
            adjacency_masks,
            result_queue,
        ),
    )

    process.start()

    process.join(
        timeout_seconds
    )

    if process.is_alive():

        process.terminate()
        process.join()

        result = {
            "status": "timeout",
            "value": None,
            "elapsed_seconds": float(
                timeout_seconds
            ),
            "cache_states": None,
            "initial_memory_mb": None,
            "final_memory_mb": None,
            "error": (
                f"Exceeded "
                f"{timeout_seconds} seconds"
            ),
        }

    else:

        try:

            result = result_queue.get_nowait()

        except queue.Empty:

            result = {
                "status": "error",
                "value": None,
                "elapsed_seconds": None,
                "cache_states": None,
                "initial_memory_mb": None,
                "final_memory_mb": None,
                "error": (
                    "Child process returned "
                    "no result"
                ),
            }

    result_queue.close()
    result_queue.join_thread()

    del process
    del result_queue

    gc.collect()

    return result


# ------------------------------------------------------------
# 8. Select six representative pilot graphs
# ------------------------------------------------------------

df_primary_pilot = pd.read_csv(
    PRIMARY_MANIFEST_PATH,
    dtype={
        "graph_id": "int32",
        "signature": "string",
        "order": "int32",
        "generator_count": "int8",
        "generator_string": "string",
        "order_group": "string",
    },
)

pilot_rows = []

for k_value in [2, 3, 4]:

    current_group = (
        df_primary_pilot.loc[
            df_primary_pilot[
                "generator_count"
            ] == k_value
        ]
        .sort_values(
            [
                "order",
                "graph_id",
            ]
        )
    )

    # One lower-order and one upper-order graph per k
    pilot_rows.append(
        current_group.iloc[0]
    )

    pilot_rows.append(
        current_group.iloc[-1]
    )

df_pilot_graphs = pd.DataFrame(
    pilot_rows
).reset_index(drop=True)


# ------------------------------------------------------------
# 9. Execute pilot
# ------------------------------------------------------------

pilot_results = []

print("=" * 90)
print("EXACT-LABELING PILOT")
print("=" * 90)

for pilot_position, row in (
    df_pilot_graphs.iterrows()
):

    graph_id = int(
        row["graph_id"]
    )

    order = int(
        row["order"]
    )

    k_value = int(
        row["generator_count"]
    )

    generators = (
        parse_generator_string(
            row["generator_string"]
        )
    )

    adjacency_masks = (
        build_adjacency_masks(
            order=order,
            generators=generators,
        )
    )

    print(
        f"\nGraph "
        f"{pilot_position + 1}/"
        f"{len(df_pilot_graphs)} | "
        f"ID={graph_id} | "
        f"k={k_value} | "
        f"n={order} | "
        f"generators={generators}",
        flush=True,
    )

    ms_result = (
        run_exact_index_isolated(
            index_name="MS",
            adjacency_masks=(
                adjacency_masks
            ),
            timeout_seconds=(
                PILOT_TIMEOUT_SECONDS
            ),
        )
    )

    print(
        "  MS:",
        ms_result["status"],
        "| time:",
        ms_result[
            "elapsed_seconds"
        ],
        "| states:",
        ms_result["cache_states"],
        flush=True,
    )

    z_result = (
        run_exact_index_isolated(
            index_name="Z",
            adjacency_masks=(
                adjacency_masks
            ),
            timeout_seconds=(
                PILOT_TIMEOUT_SECONDS
            ),
        )
    )

    print(
        "  Z :",
        z_result["status"],
        "| time:",
        z_result[
            "elapsed_seconds"
        ],
        "| states:",
        z_result["cache_states"],
        flush=True,
    )

    pilot_results.append({
        "graph_id": graph_id,
        "signature": str(
            row["signature"]
        ),
        "k": k_value,
        "order": order,
        "generators": ",".join(
            str(value)
            for value in generators
        ),
        "MS_status": (
            ms_result["status"]
        ),
        "MS": ms_result["value"],
        "MS_time_seconds": (
            ms_result[
                "elapsed_seconds"
            ]
        ),
        "MS_cache_states": (
            ms_result[
                "cache_states"
            ]
        ),
        "MS_final_memory_mb": (
            ms_result[
                "final_memory_mb"
            ]
        ),
        "Z_status": (
            z_result["status"]
        ),
        "Z": z_result["value"],
        "Z_time_seconds": (
            z_result[
                "elapsed_seconds"
            ]
        ),
        "Z_cache_states": (
            z_result[
                "cache_states"
            ]
        ),
        "Z_final_memory_mb": (
            z_result[
                "final_memory_mb"
            ]
        ),
    })

    del adjacency_masks
    del ms_result
    del z_result

    gc.collect()


# ------------------------------------------------------------
# 10. Save and report pilot results
# ------------------------------------------------------------

df_pilot_results = pd.DataFrame(
    pilot_results
)

df_pilot_results.to_csv(
    PILOT_RESULTS_PATH,
    index=False,
)

print("\n" + "=" * 90)
print("PILOT SUMMARY")
print("=" * 90)

display_columns = [
    "graph_id",
    "k",
    "order",
    "MS_status",
    "MS_time_seconds",
    "MS_cache_states",
    "Z_status",
    "Z_time_seconds",
    "Z_cache_states",
]

print(
    df_pilot_results[
        display_columns
    ].to_string(
        index=False
    )
)

print(
    "\nMS completed:",
    int(
        (
            df_pilot_results[
                "MS_status"
            ] == "completed"
        ).sum()
    ),
    "/",
    len(df_pilot_results),
)

print(
    "Z completed:",
    int(
        (
            df_pilot_results[
                "Z_status"
            ] == "completed"
        ).sum()
    ),
    "/",
    len(df_pilot_results),
)

print(
    "\nPilot results saved to:"
)

print(
    PILOT_RESULTS_PATH
)

print_memory_status(
    "Parent-process RAM after pilot"
)

del df_primary_pilot
del df_pilot_graphs
gc.collect()

print(
    "\nStep 6 completed successfully."
)

EXACT-LABELING PILOT

Graph 1/6 | ID=0 | k=2 | n=16 | generators=(1, 6)
  MS: completed | time: 0.0008018600000241349 | states: 147
  Z : completed | time: 0.013594304999969609 | states: 6118

Graph 2/6 | ID=164 | k=2 | n=26 | generators=(9, 12)
  MS: completed | time: 0.004384060000006684 | states: 1775
  Z : completed | time: 3.121250148999934 | states: 989058

Graph 3/6 | ID=50 | k=3 | n=16 | generators=(1, 3, 4)
  MS: completed | time: 0.0007825060000641315 | states: 107
  Z : completed | time: 0.021244609999939712 | states: 8759

Graph 4/6 | ID=179 | k=3 | n=26 | generators=(9, 10, 12)
  MS: completed | time: 0.0023665109999910783 | states: 1016
  Z : completed | time: 13.086743647000048 | states: 3001013

Graph 5/6 | ID=105 | k=4 | n=16 | generators=(1, 2, 3, 5)
  MS: completed | time: 0.0006639700000050652 | states: 67
  Z : completed | time: 0.029227911000020868 | states: 10824

Graph 6/6 | ID=199 | k=4 | n=26 | generators=(5, 8, 11, 12)
  MS: completed | time: 0.00230947099998

In [ ]:
# ============================================================
# STEP 7: Full memory-safe exact labeling with resume support
# ============================================================

import json
import time
import gc
from pathlib import Path

import networkx as nx
import pandas as pd


# ------------------------------------------------------------
# 1. Labeling configuration
# ------------------------------------------------------------

FULL_LABEL_TIMEOUT_SECONDS = 90

GRAPH_LABEL_CHECKPOINT_DIR = (
    CHECKPOINT_DIR
    / "circulant_graph_labels"
)

GRAPH_LABEL_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FULL_LABELS_PATH = (
    PROCESSED_DATA_DIR
    / "circulant_200_exact_labels.csv"
)

LABELING_STATUS_PATH = (
    TABLES_DIR
    / "circulant_labeling_status.csv"
)


# ------------------------------------------------------------
# 2. Load the primary benchmark manifest
# ------------------------------------------------------------

df_label_manifest = pd.read_csv(
    PRIMARY_MANIFEST_PATH,
    dtype={
        "graph_id": "int32",
        "signature": "string",
        "order": "int32",
        "generator_count": "int8",
        "generator_string": "string",
        "source_file": "string",
        "order_group": "string",
        "selection_status": "string",
    },
)

assert len(df_label_manifest) == 200
assert df_label_manifest["graph_id"].nunique() == 200


# ------------------------------------------------------------
# 3. Checkpoint utilities
# ------------------------------------------------------------

def graph_checkpoint_path(graph_id):
    """
    Return the checkpoint path for one graph.
    """

    return (
        GRAPH_LABEL_CHECKPOINT_DIR
        / f"graph_{int(graph_id):04d}.json"
    )


def load_graph_checkpoint(graph_id):
    """
    Read one existing checkpoint, if available.
    """

    checkpoint_path = graph_checkpoint_path(
        graph_id
    )

    if not checkpoint_path.exists():
        return None

    try:

        with open(
            checkpoint_path,
            "r",
            encoding="utf-8",
        ) as file_handle:

            return json.load(
                file_handle
            )

    except (
        json.JSONDecodeError,
        OSError,
    ):

        return None


def save_graph_checkpoint(
    graph_id,
    record,
):
    """
    Save one graph checkpoint atomically.

    Writing to a temporary file first prevents partially
    written checkpoints if execution is interrupted.
    """

    checkpoint_path = graph_checkpoint_path(
        graph_id
    )

    temporary_path = checkpoint_path.with_suffix(
        ".tmp"
    )

    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as file_handle:

        json.dump(
            record,
            file_handle,
            indent=2,
        )

    temporary_path.replace(
        checkpoint_path
    )


def checkpoint_is_complete(record):
    """
    A graph is complete only when all six indices exist and
    both expensive indices finished successfully.
    """

    if record is None:
        return False

    required_fields = [
        "W",
        "MS",
        "Z",
        "M1",
        "M2",
        "R",
    ]

    return (
        record.get("MS_status")
        == "completed"
        and record.get("Z_status")
        == "completed"
        and all(
            record.get(field)
            is not None
            for field in required_fields
        )
    )


# ------------------------------------------------------------
# 4. Label one graph
# ------------------------------------------------------------

def label_one_circulant_graph(
    row,
    timeout_seconds,
):
    """
    Compute all six exact topological indices for one graph.

    MS and Z are executed in isolated child processes.
    """

    graph_id = int(
        row["graph_id"]
    )

    order = int(
        row["order"]
    )

    k_value = int(
        row["generator_count"]
    )

    generators = parse_generator_string(
        row["generator_string"]
    )

    total_start = time.perf_counter()

    # Build a small NetworkX graph only for the four fast indices.
    graph = build_circulant_graph(
        order=order,
        generators=generators,
    )

    if not nx.is_connected(graph):

        raise ValueError(
            f"Graph {graph_id} is disconnected."
        )

    fast_start = time.perf_counter()

    W = compute_wiener_index(
        graph
    )

    M1 = compute_first_zagreb_index(
        graph
    )

    M2 = compute_second_zagreb_index(
        graph
    )

    R = compute_randic_index(
        graph
    )

    fast_elapsed = (
        time.perf_counter()
        - fast_start
    )

    # The recursive routines use only compact adjacency masks.
    adjacency_masks = build_adjacency_masks(
        order=order,
        generators=generators,
    )

    # NetworkX graph is no longer required.
    del graph
    gc.collect()

    ms_result = run_exact_index_isolated(
        index_name="MS",
        adjacency_masks=adjacency_masks,
        timeout_seconds=timeout_seconds,
    )

    z_result = run_exact_index_isolated(
        index_name="Z",
        adjacency_masks=adjacency_masks,
        timeout_seconds=timeout_seconds,
    )

    total_elapsed = (
        time.perf_counter()
        - total_start
    )

    record = {
        "graph_id": graph_id,
        "signature": str(
            row["signature"]
        ),
        "family": (
            f"circulant_k{k_value}"
        ),
        "generator_count": k_value,
        "order": order,
        "generators": list(
            generators
        ),
        "order_group": str(
            row["order_group"]
        ),
        "W": int(W),
        "MS": (
            str(ms_result["value"])
            if ms_result["value"] is not None
            else None
        ),
        "Z": (
            str(z_result["value"])
            if z_result["value"] is not None
            else None
        ),
        "M1": int(M1),
        "M2": int(M2),
        "R": float(R),
        "MS_status": (
            ms_result["status"]
        ),
        "Z_status": (
            z_result["status"]
        ),
        "fast_indices_time_seconds": float(
            fast_elapsed
        ),
        "MS_time_seconds": (
            float(
                ms_result[
                    "elapsed_seconds"
                ]
            )
            if ms_result[
                "elapsed_seconds"
            ] is not None
            else None
        ),
        "Z_time_seconds": (
            float(
                z_result[
                    "elapsed_seconds"
                ]
            )
            if z_result[
                "elapsed_seconds"
            ] is not None
            else None
        ),
        "total_time_seconds": float(
            total_elapsed
        ),
        "MS_cache_states": (
            int(
                ms_result[
                    "cache_states"
                ]
            )
            if ms_result[
                "cache_states"
            ] is not None
            else None
        ),
        "Z_cache_states": (
            int(
                z_result[
                    "cache_states"
                ]
            )
            if z_result[
                "cache_states"
            ] is not None
            else None
        ),
        "MS_error": (
            ms_result["error"]
        ),
        "Z_error": (
            z_result["error"]
        ),
    }

    del adjacency_masks
    del ms_result
    del z_result
    gc.collect()

    return record


# ------------------------------------------------------------
# 5. Process graphs from lower to higher order
# ------------------------------------------------------------

df_label_manifest = (
    df_label_manifest
    .sort_values(
        [
            "order",
            "generator_count",
            "graph_id",
        ]
    )
    .reset_index(drop=True)
)

already_completed = 0
newly_completed = 0
incomplete = 0
errors = 0

session_start = time.perf_counter()

print("=" * 96)
print("FULL EXACT LABELING OF 200 CIRCULANT GRAPHS")
print("=" * 96)
print(
    f"Timeout per expensive index: "
    f"{FULL_LABEL_TIMEOUT_SECONDS} seconds"
)
print(
    f"Checkpoint directory: "
    f"{GRAPH_LABEL_CHECKPOINT_DIR}"
)

for position, row in df_label_manifest.iterrows():

    graph_id = int(
        row["graph_id"]
    )

    existing_record = load_graph_checkpoint(
        graph_id
    )

    if checkpoint_is_complete(
        existing_record
    ):

        already_completed += 1

        print(
            f"[{position + 1:03d}/200] "
            f"ID={graph_id:03d} | "
            f"already completed — skipped",
            flush=True,
        )

        continue

    print(
        f"\n[{position + 1:03d}/200] "
        f"ID={graph_id:03d} | "
        f"k={int(row['generator_count'])} | "
        f"n={int(row['order'])} | "
        f"{row['signature']}",
        flush=True,
    )

    try:

        record = label_one_circulant_graph(
            row=row,
            timeout_seconds=(
                FULL_LABEL_TIMEOUT_SECONDS
            ),
        )

        save_graph_checkpoint(
            graph_id=graph_id,
            record=record,
        )

        is_complete = checkpoint_is_complete(
            record
        )

        if is_complete:

            newly_completed += 1

        else:

            incomplete += 1

        print(
            f"  MS: {record['MS_status']} "
            f"({record['MS_time_seconds']:.3f} s)"
            if record["MS_time_seconds"] is not None
            else
            f"  MS: {record['MS_status']}",
            flush=True,
        )

        print(
            f"  Z : {record['Z_status']} "
            f"({record['Z_time_seconds']:.3f} s)"
            if record["Z_time_seconds"] is not None
            else
            f"  Z : {record['Z_status']}",
            flush=True,
        )

        print(
            f"  Total: "
            f"{record['total_time_seconds']:.3f} s",
            flush=True,
        )

    except Exception as exception:

        errors += 1

        error_record = {
            "graph_id": graph_id,
            "signature": str(
                row["signature"]
            ),
            "family": (
                f"circulant_k"
                f"{int(row['generator_count'])}"
            ),
            "generator_count": int(
                row["generator_count"]
            ),
            "order": int(
                row["order"]
            ),
            "generators": list(
                parse_generator_string(
                    row["generator_string"]
                )
            ),
            "order_group": str(
                row["order_group"]
            ),
            "W": None,
            "MS": None,
            "Z": None,
            "M1": None,
            "M2": None,
            "R": None,
            "MS_status": "error",
            "Z_status": "error",
            "error": repr(
                exception
            ),
        }

        save_graph_checkpoint(
            graph_id=graph_id,
            record=error_record,
        )

        print(
            "  ERROR:",
            repr(exception),
            flush=True,
        )

    # This releases only unreachable temporary Python objects.
    # It does not restart or disconnect the Colab runtime.
    gc.collect()

    if (
        (position + 1) % 10 == 0
        or position + 1 == len(
            df_label_manifest
        )
    ):

        elapsed_minutes = (
            time.perf_counter()
            - session_start
        ) / 60.0

        memory = get_memory_status()

        print(
            f"\n--- Progress checkpoint: "
            f"{position + 1}/200 | "
            f"elapsed={elapsed_minutes:.1f} min | "
            f"available RAM="
            f"{memory['system_ram_available_gb']:.2f} GB "
            f"---",
            flush=True,
        )


# ------------------------------------------------------------
# 6. Aggregate all individual checkpoints
# ------------------------------------------------------------

all_checkpoint_records = []

for _, row in (
    df_label_manifest
    .sort_values("graph_id")
    .iterrows()
):

    graph_id = int(
        row["graph_id"]
    )

    record = load_graph_checkpoint(
        graph_id
    )

    if record is not None:

        all_checkpoint_records.append(
            record
        )

df_all_labels = pd.DataFrame(
    all_checkpoint_records
)

if not df_all_labels.empty:

    df_all_labels = (
        df_all_labels
        .sort_values("graph_id")
        .reset_index(drop=True)
    )

    df_all_labels.to_csv(
        FULL_LABELS_PATH,
        index=False,
    )


# ------------------------------------------------------------
# 7. Create labeling-status table
# ------------------------------------------------------------

status_rows = []

for k_value in [2, 3, 4]:

    current = df_all_labels.loc[
        df_all_labels[
            "generator_count"
        ] == k_value
    ]

    status_rows.append({
        "generator_dimension": (
            f"k={k_value}"
        ),
        "attempted_graphs": int(
            len(current)
        ),
        "fully_completed": int(
            current.apply(
                lambda row: (
                    row.get(
                        "MS_status"
                    ) == "completed"
                    and row.get(
                        "Z_status"
                    ) == "completed"
                ),
                axis=1,
            ).sum()
        ),
        "MS_completed": int(
            (
                current[
                    "MS_status"
                ] == "completed"
            ).sum()
        ),
        "Z_completed": int(
            (
                current[
                    "Z_status"
                ] == "completed"
            ).sum()
        ),
        "MS_timeouts": int(
            (
                current[
                    "MS_status"
                ] == "timeout"
            ).sum()
        ),
        "Z_timeouts": int(
            (
                current[
                    "Z_status"
                ] == "timeout"
            ).sum()
        ),
    })

df_labeling_status = pd.DataFrame(
    status_rows
)

df_labeling_status.to_csv(
    LABELING_STATUS_PATH,
    index=False,
)


# ------------------------------------------------------------
# 8. Final report
# ------------------------------------------------------------

fully_completed_mask = (
    (
        df_all_labels["MS_status"]
        == "completed"
    )
    &
    (
        df_all_labels["Z_status"]
        == "completed"
    )
)

fully_completed_count = int(
    fully_completed_mask.sum()
)

print("\n" + "=" * 96)
print("FULL LABELING SESSION SUMMARY")
print("=" * 96)

print(
    "Checkpoint records:",
    len(df_all_labels),
)

print(
    "Already completed before this run:",
    already_completed,
)

print(
    "Newly completed:",
    newly_completed,
)

print(
    "Fully completed graphs:",
    fully_completed_count,
    "/ 200",
)

print(
    "Incomplete graphs:",
    200 - fully_completed_count,
)

print(
    "Execution errors:",
    errors,
)

print("\nStatus by generator dimension:")
print(
    df_labeling_status.to_string(
        index=False
    )
)

print(
    "\nAggregated labels saved to:"
)
print(
    FULL_LABELS_PATH
)

print(
    "Labeling status saved to:"
)
print(
    LABELING_STATUS_PATH
)

print_memory_status(
    "Parent-process RAM after full labeling"
)

print(
    "\nStep 7 finished. "
    "The cell may be rerun safely to resume incomplete work."
)

FULL EXACT LABELING OF 200 CIRCULANT GRAPHS
Timeout per expensive index: 90 seconds
Checkpoint directory: C:\Unicyclic_Bicyclic_Experiment_paper\Larger graph/checkpoints/circulant_graph_labels

[001/200] ID=000 | k=2 | n=16 | C(16; 1,6)
  MS: completed (0.001 s)
  Z : completed (0.014 s)
  Total: 0.252 s

[002/200] ID=001 | k=2 | n=16 | C(16; 2,3)
  MS: completed (0.001 s)
  Z : completed (0.015 s)
  Total: 0.232 s

[003/200] ID=002 | k=2 | n=16 | C(16; 2,5)
  MS: completed (0.001 s)
  Z : completed (0.012 s)
  Total: 0.226 s

[004/200] ID=003 | k=2 | n=16 | C(16; 6,7)
  MS: completed (0.001 s)
  Z : completed (0.013 s)
  Total: 0.235 s

[005/200] ID=050 | k=3 | n=16 | C(16; 1,3,4)
  MS: completed (0.001 s)
  Z : completed (0.021 s)
  Total: 0.243 s

[006/200] ID=051 | k=3 | n=16 | C(16; 2,3,4)
  MS: completed (0.001 s)
  Z : completed (0.023 s)
  Total: 0.268 s

[007/200] ID=052 | k=3 | n=16 | C(16; 2,4,5)
  MS: completed (0.001 s)
  Z : completed (0.024 s)
  Total: 0.262 s

[008/200]

In [ ]:
# ============================================================
# STEP 8: Validate labels and prepare final modeling dataset
# ============================================================

import ast
import math
import gc

import numpy as np
import pandas as pd
import networkx as nx


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

MODELING_DATA_PATH = (
    PROCESSED_DATA_DIR
    / "circulant_200_modeling_dataset.csv"
)

TARGET_SUMMARY_PATH = (
    TABLES_DIR
    / "circulant_target_summary.csv"
)

FINAL_COMPOSITION_PATH = (
    TABLES_DIR
    / "circulant_final_benchmark_composition.csv"
)


# ------------------------------------------------------------
# 2. Load exact labels and benchmark manifest
# ------------------------------------------------------------

df_labels = pd.read_csv(
    FULL_LABELS_PATH,
    dtype={
        "graph_id": "int32",
        "signature": "string",
        "family": "string",
        "generator_count": "int8",
        "order": "int32",
        "order_group": "string",
        "MS_status": "string",
        "Z_status": "string",
    },
)

df_manifest = pd.read_csv(
    PRIMARY_MANIFEST_PATH,
    dtype={
        "graph_id": "int32",
        "signature": "string",
        "order": "int32",
        "generator_count": "int8",
        "generator_string": "string",
        "source_file": "string",
        "order_group": "string",
    },
)

assert len(df_labels) == 200
assert len(df_manifest) == 200
assert df_labels["graph_id"].nunique() == 200
assert df_manifest["graph_id"].nunique() == 200


# ------------------------------------------------------------
# 3. Validate labeling status
# ------------------------------------------------------------

assert (
    df_labels["MS_status"] == "completed"
).all()

assert (
    df_labels["Z_status"] == "completed"
).all()


# ------------------------------------------------------------
# 4. Convert target columns to numeric types
# ------------------------------------------------------------

target_columns = [
    "W",
    "MS",
    "Z",
    "M1",
    "M2",
    "R",
]

for column in target_columns:

    df_labels[column] = pd.to_numeric(
        df_labels[column],
        errors="raise",
    )

# Preserve exact integer-valued indices
for column in [
    "W",
    "MS",
    "Z",
    "M1",
    "M2",
]:
    df_labels[column] = (
        df_labels[column]
        .astype(object)
    )

df_labels["R"] = (
    df_labels["R"]
    .astype("float64")
)


# ------------------------------------------------------------
# 5. Merge metadata and labels
# ------------------------------------------------------------

df_modeling = df_manifest.merge(
    df_labels[
        [
            "graph_id",
            "W",
            "MS",
            "Z",
            "M1",
            "M2",
            "R",
            "MS_time_seconds",
            "Z_time_seconds",
            "total_time_seconds",
        ]
    ],
    on="graph_id",
    how="inner",
    validate="one_to_one",
)

assert len(df_modeling) == 200


# ------------------------------------------------------------
# 6. Parse generator tuples
# ------------------------------------------------------------

def parse_generators_from_string(
    generator_string,
):
    return tuple(
        int(value.strip())
        for value in str(
            generator_string
        ).split(",")
        if value.strip()
    )


# ------------------------------------------------------------
# 7. Compute structural descriptors one graph at a time
# ------------------------------------------------------------

descriptor_rows = []

for position, row in df_modeling.iterrows():

    graph_id = int(
        row["graph_id"]
    )

    order = int(
        row["order"]
    )

    generators = (
        parse_generators_from_string(
            row["generator_string"]
        )
    )

    graph = build_circulant_graph(
        order=order,
        generators=generators,
    )

    degrees = np.fromiter(
        (
            degree
            for _, degree
            in graph.degree()
        ),
        dtype=np.float64,
    )

    edge_count = int(
        graph.number_of_edges()
    )

    average_degree = float(
        degrees.mean()
    )

    degree_variance = float(
        degrees.var()
    )

    maximum_degree = int(
        degrees.max()
    )

    minimum_degree = int(
        degrees.min()
    )

    density = float(
        nx.density(graph)
    )

    diameter = int(
        nx.diameter(graph)
    )

    average_shortest_path = float(
        nx.average_shortest_path_length(
            graph
        )
    )

    clustering = float(
        nx.average_clustering(
            graph
        )
    )

    cyclomatic_number = int(
        edge_count
        - order
        + 1
    )

    descriptor_rows.append({
        "graph_id": graph_id,
        "edge_count": edge_count,
        "average_degree": average_degree,
        "degree_variance": degree_variance,
        "maximum_degree": maximum_degree,
        "minimum_degree": minimum_degree,
        "density": density,
        "diameter": diameter,
        "average_shortest_path": (
            average_shortest_path
        ),
        "average_clustering": clustering,
        "cyclomatic_number": (
            cyclomatic_number
        ),
    })

    del graph
    del degrees

    if (
        (position + 1) % 25 == 0
        or position + 1 == len(
            df_modeling
        )
    ):

        print(
            f"Computed descriptors for "
            f"{position + 1}/"
            f"{len(df_modeling)} graphs",
            flush=True,
        )

        gc.collect()


df_descriptors = pd.DataFrame(
    descriptor_rows
)

df_modeling = df_modeling.merge(
    df_descriptors,
    on="graph_id",
    how="inner",
    validate="one_to_one",
)


# ------------------------------------------------------------
# 8. Add generator-level descriptors
# ------------------------------------------------------------

df_modeling[
    "generator_sum"
] = df_modeling[
    "generator_string"
].apply(
    lambda value: sum(
        parse_generators_from_string(
            value
        )
    )
)

df_modeling[
    "generator_mean"
] = df_modeling[
    "generator_string"
].apply(
    lambda value: float(
        np.mean(
            parse_generators_from_string(
                value
            )
        )
    )
)

df_modeling[
    "generator_std"
] = df_modeling[
    "generator_string"
].apply(
    lambda value: float(
        np.std(
            parse_generators_from_string(
                value
            )
        )
    )
)

df_modeling[
    "generator_min"
] = df_modeling[
    "generator_string"
].apply(
    lambda value: min(
        parse_generators_from_string(
            value
        )
    )
)

df_modeling[
    "generator_max"
] = df_modeling[
    "generator_string"
].apply(
    lambda value: max(
        parse_generators_from_string(
            value
        )
    )
)


# ------------------------------------------------------------
# 9. Validate final dataset
# ------------------------------------------------------------

assert len(df_modeling) == 200
assert df_modeling["graph_id"].nunique() == 200

for target in target_columns:

    assert df_modeling[
        target
    ].notna().all()

    assert (
        df_modeling[
            target
        ].astype(float)
        > 0
    ).all()

assert (
    df_modeling[
        "order_group"
    ].value_counts().to_dict()
    == {
        "development": 155,
        "heldout": 45,
    }
)


# ------------------------------------------------------------
# 10. Save final modeling dataset
# ------------------------------------------------------------

df_modeling = (
    df_modeling
    .sort_values("graph_id")
    .reset_index(drop=True)
)

df_modeling.to_csv(
    MODELING_DATA_PATH,
    index=False,
)


# ------------------------------------------------------------
# 11. Target summary
# ------------------------------------------------------------

target_summary_rows = []

for target in target_columns:

    values = (
        df_modeling[target]
        .astype(float)
    )

    target_summary_rows.append({
        "target": target,
        "minimum": values.min(),
        "maximum": values.max(),
        "mean": values.mean(),
        "standard_deviation": (
            values.std(ddof=1)
        ),
        "median": values.median(),
    })

df_target_summary = pd.DataFrame(
    target_summary_rows
)

df_target_summary.to_csv(
    TARGET_SUMMARY_PATH,
    index=False,
)


# ------------------------------------------------------------
# 12. Final benchmark composition table
# ------------------------------------------------------------

composition_rows = []

for k_value in [2, 3, 4]:

    current = df_modeling.loc[
        df_modeling[
            "generator_count"
        ] == k_value
    ]

    composition_rows.append({
        "generator_dimension": (
            f"k={k_value}"
        ),
        "graphs": int(
            len(current)
        ),
        "minimum_order": int(
            current["order"].min()
        ),
        "maximum_order": int(
            current["order"].max()
        ),
        "unique_orders": int(
            current["order"].nunique()
        ),
        "development_graphs": int(
            (
                current[
                    "order_group"
                ] == "development"
            ).sum()
        ),
        "heldout_graphs": int(
            (
                current[
                    "order_group"
                ] == "heldout"
            ).sum()
        ),
    })

composition_rows.append({
    "generator_dimension": "Total",
    "graphs": int(
        len(df_modeling)
    ),
    "minimum_order": int(
        df_modeling["order"].min()
    ),
    "maximum_order": int(
        df_modeling["order"].max()
    ),
    "unique_orders": int(
        df_modeling["order"].nunique()
    ),
    "development_graphs": int(
        (
            df_modeling[
                "order_group"
            ] == "development"
        ).sum()
    ),
    "heldout_graphs": int(
        (
            df_modeling[
                "order_group"
            ] == "heldout"
        ).sum()
    ),
})

df_final_composition = pd.DataFrame(
    composition_rows
)

df_final_composition.to_csv(
    FINAL_COMPOSITION_PATH,
    index=False,
)


# ------------------------------------------------------------
# 13. Report
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("FINAL MODELING DATASET")
print("=" * 88)

print(
    "Dataset shape:",
    df_modeling.shape,
)

print(
    "Development graphs:",
    int(
        (
            df_modeling[
                "order_group"
            ] == "development"
        ).sum()
    ),
)

print(
    "Held-out graphs:",
    int(
        (
            df_modeling[
                "order_group"
            ] == "heldout"
        ).sum()
    ),
)

print("\nFinal benchmark composition:")
print(
    df_final_composition.to_string(
        index=False
    )
)

print("\nTarget summary:")
print(
    df_target_summary.to_string(
        index=False
    )
)

print(
    "\nModeling dataset saved to:"
)
print(
    MODELING_DATA_PATH
)

print(
    "Target summary saved to:"
)
print(
    TARGET_SUMMARY_PATH
)

print(
    "Final composition saved to:"
)
print(
    FINAL_COMPOSITION_PATH
)

print_memory_status(
    "RAM status after dataset preparation"
)

gc.collect()

print(
    "\nStep 8 completed successfully."
)

Computed descriptors for 25/200 graphs
Computed descriptors for 50/200 graphs
Computed descriptors for 75/200 graphs
Computed descriptors for 100/200 graphs
Computed descriptors for 125/200 graphs
Computed descriptors for 150/200 graphs
Computed descriptors for 175/200 graphs
Computed descriptors for 200/200 graphs

FINAL MODELING DATASET
Dataset shape: (200, 32)
Development graphs: 155
Held-out graphs: 45

Final benchmark composition:
generator_dimension  graphs  minimum_order  maximum_order  unique_orders  development_graphs  heldout_graphs
                k=2      60             16             26             11                  50              10
                k=3      70             16             26             11                  55              15
                k=4      70             16             26             11                  50              20
              Total     200             16             26             11                 155              45

Target summary

In [ ]:
# ============================================================
# STEP 9: Build the compact PyTorch Geometric dataset
# ============================================================

# Install PyTorch Geometric only if it is unavailable
try:
    import torch_geometric
except ImportError:
    !pip -q install torch-geometric

import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import torch

from torch_geometric.data import Data


# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------

GLOBAL_SEED = 42

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        GLOBAL_SEED
    )

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ------------------------------------------------------------
# 2. Paths
# ------------------------------------------------------------

PYG_DATASET_PATH = (
    PROCESSED_DATA_DIR
    / "circulant_200_pyg_dataset.pt"
)

PYG_METADATA_PATH = (
    PROCESSED_DATA_DIR
    / "circulant_200_pyg_metadata.csv"
)

NODE_FEATURE_SUMMARY_PATH = (
    TABLES_DIR
    / "circulant_node_feature_summary.csv"
)


# ------------------------------------------------------------
# 3. Load final modeling dataset
# ------------------------------------------------------------

df_modeling_pyg = pd.read_csv(
    MODELING_DATA_PATH,
    dtype={
        "graph_id": "int32",
        "signature": "string",
        "order": "int32",
        "generator_count": "int8",
        "generator_string": "string",
        "order_group": "string",
    },
)

assert len(df_modeling_pyg) == 200

TARGET_COLUMNS = [
    "W",
    "MS",
    "Z",
    "M1",
    "M2",
    "R",
]

LOG_TARGETS = {"MS", "Z"}


def transform_targets_for_model(target_values):
    """
    Apply log1p only to MS(G) and Z(G).
    The remaining targets stay on their original scale.
    """
    transformed = np.asarray(
        target_values,
        dtype=np.float64,
    ).copy()

    for target_index, target_name in enumerate(
        TARGET_COLUMNS
    ):
        if target_name in LOG_TARGETS:
            transformed[..., target_index] = np.log1p(
                np.maximum(
                    transformed[..., target_index],
                    0.0,
                )
            )

    return transformed


def inverse_targets_from_model(transformed_values):
    """
    Reverse the mixed target transformation and enforce
    non-negative topological-index predictions.
    """
    original = np.asarray(
        transformed_values,
        dtype=np.float64,
    ).copy()

    for target_index, target_name in enumerate(
        TARGET_COLUMNS
    ):
        if target_name in LOG_TARGETS:
            original[..., target_index] = np.expm1(
                original[..., target_index]
            )

    return np.maximum(
        original,
        0.0,
    )

NODE_FEATURE_NAMES = [
    "degree_normalized",
    "clustering_coefficient",
    "betweenness_centrality",
    "closeness_centrality",
    "eigenvector_centrality",
]


# ------------------------------------------------------------
# 4. Node-feature extraction
# ------------------------------------------------------------

def extract_node_features(graph):
    """
    Compute five node-level structural descriptors.

    No dataset-level scaling is applied here. Feature
    standardization will be fitted only on each training split.
    """

    number_of_nodes = graph.number_of_nodes()

    degree_dictionary = dict(
        graph.degree()
    )

    clustering_dictionary = (
        nx.clustering(graph)
    )

    betweenness_dictionary = (
        nx.betweenness_centrality(
            graph,
            normalized=True,
        )
    )

    closeness_dictionary = (
        nx.closeness_centrality(
            graph
        )
    )

    eigenvector_dictionary = (
        nx.eigenvector_centrality_numpy(
            graph
        )
    )

    feature_rows = []

    normalization_denominator = max(
        number_of_nodes - 1,
        1,
    )

    for node in range(number_of_nodes):

        feature_rows.append([
            degree_dictionary[node]
            / normalization_denominator,
            clustering_dictionary[node],
            betweenness_dictionary[node],
            closeness_dictionary[node],
            eigenvector_dictionary[node],
        ])

    return np.asarray(
        feature_rows,
        dtype=np.float32,
    )


# ------------------------------------------------------------
# 5. Edge-index conversion
# ------------------------------------------------------------

def graph_to_edge_index(graph):
    """
    Convert an undirected NetworkX graph into a bidirectional
    PyG edge-index tensor.
    """

    directed_edges = []

    for source, target in graph.edges():

        directed_edges.append(
            [source, target]
        )

        directed_edges.append(
            [target, source]
        )

    edge_array = np.asarray(
        directed_edges,
        dtype=np.int64,
    ).T

    return torch.tensor(
        edge_array,
        dtype=torch.long,
    )


# ------------------------------------------------------------
# 6. Build one compact PyG object at a time
# ------------------------------------------------------------

pyg_dataset = []
metadata_rows = []
all_node_features = []

print("=" * 86)
print("BUILDING PYTORCH GEOMETRIC DATASET")
print("=" * 86)

for position, row in df_modeling_pyg.iterrows():

    graph_id = int(
        row["graph_id"]
    )

    order = int(
        row["order"]
    )

    generators = (
        parse_generators_from_string(
            row["generator_string"]
        )
    )

    graph = build_circulant_graph(
        order=order,
        generators=generators,
    )

    node_features = extract_node_features(
        graph
    )

    edge_index = graph_to_edge_index(
        graph
    )

    original_targets = np.asarray(
        [
            float(row[target])
            for target in TARGET_COLUMNS
        ],
        dtype=np.float64,
    )

    transformed_targets = transform_targets_for_model(
        original_targets
    ).astype(
        np.float32
    )

    data_object = Data(
        x=torch.tensor(
            node_features,
            dtype=torch.float32,
        ),
        edge_index=edge_index,
        y=torch.tensor(
            transformed_targets,
            dtype=torch.float32,
        ).view(1, -1),
    )

    # Retain lightweight graph metadata
    data_object.graph_id = graph_id
    data_object.order = order
    data_object.generator_count = int(
        row["generator_count"]
    )

    pyg_dataset.append(
        data_object
    )

    all_node_features.append(
        node_features
    )

    metadata_rows.append({
        "dataset_index": position,
        "graph_id": graph_id,
        "signature": str(
            row["signature"]
        ),
        "order": order,
        "generator_count": int(
            row["generator_count"]
        ),
        "generator_string": str(
            row["generator_string"]
        ),
        "order_group": str(
            row["order_group"]
        ),
        "node_count": int(
            graph.number_of_nodes()
        ),
        "edge_count": int(
            graph.number_of_edges()
        ),
    })

    del graph
    del node_features
    del edge_index
    del original_targets
    del transformed_targets
    del data_object

    if (
        (position + 1) % 25 == 0
        or position + 1
        == len(df_modeling_pyg)
    ):

        print(
            f"Prepared "
            f"{position + 1}/"
            f"{len(df_modeling_pyg)} graphs",
            flush=True,
        )

        gc.collect()


# ------------------------------------------------------------
# 7. Save PyG dataset and metadata
# ------------------------------------------------------------

torch.save(
    pyg_dataset,
    PYG_DATASET_PATH,
)

df_pyg_metadata = pd.DataFrame(
    metadata_rows
)

df_pyg_metadata.to_csv(
    PYG_METADATA_PATH,
    index=False,
)


# ------------------------------------------------------------
# 8. Node-feature summary
# ------------------------------------------------------------

node_feature_matrix = np.concatenate(
    all_node_features,
    axis=0,
)

feature_summary_rows = []

for feature_index, feature_name in enumerate(
    NODE_FEATURE_NAMES
):

    values = node_feature_matrix[
        :,
        feature_index,
    ]

    feature_summary_rows.append({
        "feature": feature_name,
        "minimum": float(
            values.min()
        ),
        "maximum": float(
            values.max()
        ),
        "mean": float(
            values.mean()
        ),
        "standard_deviation": float(
            values.std(ddof=1)
        ),
    })

df_node_feature_summary = pd.DataFrame(
    feature_summary_rows
)

df_node_feature_summary.to_csv(
    NODE_FEATURE_SUMMARY_PATH,
    index=False,
)


# ------------------------------------------------------------
# 9. Validation
# ------------------------------------------------------------

assert len(pyg_dataset) == 200
assert len(df_pyg_metadata) == 200

for data_object in pyg_dataset:

    assert data_object.x.ndim == 2
    assert data_object.x.shape[1] == 5
    assert data_object.edge_index.shape[0] == 2
    assert data_object.y.shape == (1, 6)
    assert torch.isfinite(
        data_object.x
    ).all()
    assert torch.isfinite(
        data_object.y
    ).all()


# ------------------------------------------------------------
# 10. Report
# ------------------------------------------------------------

example = pyg_dataset[0]

dataset_size_mb = (
    PYG_DATASET_PATH.stat().st_size
    / (1024 ** 2)
)

print("\n" + "=" * 86)
print("PYTORCH GEOMETRIC DATASET COMPLETE")
print("=" * 86)

print(
    "Graphs:",
    len(pyg_dataset),
)

print(
    "Total nodes:",
    int(
        df_pyg_metadata[
            "node_count"
        ].sum()
    ),
)

print(
    "Total undirected edges:",
    int(
        df_pyg_metadata[
            "edge_count"
        ].sum()
    ),
)

print(
    "Node features:",
    len(NODE_FEATURE_NAMES),
)

print(
    "Targets:",
    len(TARGET_COLUMNS),
)

print(
    "Example x shape:",
    tuple(
        example.x.shape
    ),
)

print(
    "Example edge_index shape:",
    tuple(
        example.edge_index.shape
    ),
)

print(
    "Example y shape:",
    tuple(
        example.y.shape
    ),
)

print(
    "Serialized dataset size:",
    f"{dataset_size_mb:.3f} MB",
)

print("\nNode-feature summary:")
print(
    df_node_feature_summary.to_string(
        index=False
    )
)

print(
    "\nPyG dataset saved to:"
)
print(
    PYG_DATASET_PATH
)

print(
    "Metadata saved to:"
)
print(
    PYG_METADATA_PATH
)

print_memory_status(
    "RAM status after PyG dataset construction"
)

del all_node_features
del node_feature_matrix
gc.collect()

print(
    "\nStep 9 completed successfully."
)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.7 MB/s eta 0:00:00
BUILDING PYTORCH GEOMETRIC DATASET
Prepared 25/200 graphs
Prepared 50/200 graphs
Prepared 75/200 graphs
Prepared 100/200 graphs
Prepared 125/200 graphs
Prepared 150/200 graphs
Prepared 175/200 graphs
Prepared 200/200 graphs

PYTORCH GEOMETRIC DATASET COMPLETE
Graphs: 200
Total nodes: 4141
Total undirected edges: 12666
Node features: 5
Targets: 6
Example x shape: (16, 5)
Example edge_index shape: (2, 64)
Example y shape: (1, 6)
Serialized dataset size: 0.641 MB

Node-feature summary:
               feature  minimum  maximum     mean  standard_deviation
     degree_normalized 0.160000 0.533333 0.309911            0.093985
clustering_coefficient 0.000000 0.535714 0.130208            0.156250
betweenness_centrality 0.028333 0.066667 0.043124            0.013466
  closeness_centrality 0.416667 0.681818 0.560721            0.073505
eigen

In [ ]:
# ============================================================
# STEP 10: Single-seed GIN verification run
# Seed 42 | 70/15/15 stratified split
# ============================================================

import copy
import gc
import math
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from torch_geometric.loader import DataLoader
from torch_geometric.nn import (
    GINConv,
    global_mean_pool,
)


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

RUN_SEED = 42

BATCH_SIZE = 16
LEARNING_RATE = 1.0e-3
WEIGHT_DECAY = 1.0e-5
MAX_EPOCHS = 500
EARLY_STOPPING_PATIENCE = 60

HIDDEN_DIMENSION = 64
DROPOUT_RATE = 0.10

TARGET_COLUMNS = [
    "W",
    "MS",
    "Z",
    "M1",
    "M2",
    "R",
]

SINGLE_RUN_METRICS_PATH = (
    TABLES_DIR
    / "circulant_seed42_test_metrics.csv"
)

SINGLE_RUN_HISTORY_PATH = (
    RESULTS_DIR
    / "logs"
    / "circulant_seed42_training_history.csv"
)

SINGLE_RUN_MODEL_PATH = (
    MODELS_DIR
    / "circulant_seed42_best_gin.pt"
)


# ------------------------------------------------------------
# 2. Reproducibility utility
# ------------------------------------------------------------

def set_all_random_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_all_random_seeds(
    RUN_SEED
)


# ------------------------------------------------------------
# 3. Load compact PyG dataset and metadata
# ------------------------------------------------------------

try:
    pyg_dataset_loaded = torch.load(
        PYG_DATASET_PATH,
        weights_only=False,
    )
except TypeError:
    pyg_dataset_loaded = torch.load(
        PYG_DATASET_PATH
    )

df_metadata_loaded = pd.read_csv(
    PYG_METADATA_PATH,
    dtype={
        "dataset_index": "int32",
        "graph_id": "int32",
        "order": "int32",
        "generator_count": "int8",
        "order_group": "string",
    },
)

assert len(
    pyg_dataset_loaded
) == 200

assert len(
    df_metadata_loaded
) == 200


# ------------------------------------------------------------
# 4. Create a 70/15/15 stratified split
# ------------------------------------------------------------

all_indices = np.arange(
    len(pyg_dataset_loaded)
)

stratification_labels = (
    df_metadata_loaded[
        "generator_count"
    ]
    .astype(str)
    .to_numpy()
)

train_indices, temporary_indices = (
    train_test_split(
        all_indices,
        test_size=0.30,
        random_state=RUN_SEED,
        shuffle=True,
        stratify=stratification_labels,
    )
)

temporary_stratification = (
    stratification_labels[
        temporary_indices
    ]
)

validation_indices, test_indices = (
    train_test_split(
        temporary_indices,
        test_size=0.50,
        random_state=RUN_SEED,
        shuffle=True,
        stratify=temporary_stratification,
    )
)

assert len(train_indices) == 140
assert len(validation_indices) == 30
assert len(test_indices) == 30

assert (
    set(train_indices)
    .isdisjoint(validation_indices)
)

assert (
    set(train_indices)
    .isdisjoint(test_indices)
)

assert (
    set(validation_indices)
    .isdisjoint(test_indices)
)


# ------------------------------------------------------------
# 5. Compute training-only node-feature statistics
# ------------------------------------------------------------

training_node_matrix = torch.cat(
    [
        pyg_dataset_loaded[
            int(index)
        ].x
        for index in train_indices
    ],
    dim=0,
)

node_feature_mean = (
    training_node_matrix.mean(
        dim=0
    )
)

node_feature_std = (
    training_node_matrix.std(
        dim=0,
        unbiased=False,
    )
)

node_feature_std = torch.where(
    node_feature_std < 1.0e-8,
    torch.ones_like(
        node_feature_std
    ),
    node_feature_std,
)


# ------------------------------------------------------------
# 6. Compute training-only target statistics
# ------------------------------------------------------------

training_transformed_targets = torch.cat(
    [
        pyg_dataset_loaded[
            int(index)
        ].y
        for index in train_indices
    ],
    dim=0,
)

target_mean = (
    training_transformed_targets.mean(
        dim=0
    )
)

target_std = (
    training_transformed_targets.std(
        dim=0,
        unbiased=False,
    )
)

target_std = torch.where(
    target_std < 1.0e-8,
    torch.ones_like(
        target_std
    ),
    target_std,
)


# ------------------------------------------------------------
# 7. Create split-specific standardized copies
# ------------------------------------------------------------

def prepare_split_dataset(
    indices,
    original_dataset,
    feature_mean,
    feature_std,
    output_mean,
    output_std,
):
    prepared_graphs = []

    for index in indices:

        original = original_dataset[
            int(index)
        ]

        graph_data = original.clone()

        graph_data.x = (
            graph_data.x
            - feature_mean
        ) / feature_std

        graph_data.y = (
            graph_data.y
            - output_mean
        ) / output_std

        prepared_graphs.append(
            graph_data
        )

    return prepared_graphs


train_dataset = prepare_split_dataset(
    indices=train_indices,
    original_dataset=pyg_dataset_loaded,
    feature_mean=node_feature_mean,
    feature_std=node_feature_std,
    output_mean=target_mean,
    output_std=target_std,
)

validation_dataset = prepare_split_dataset(
    indices=validation_indices,
    original_dataset=pyg_dataset_loaded,
    feature_mean=node_feature_mean,
    feature_std=node_feature_std,
    output_mean=target_mean,
    output_std=target_std,
)

test_dataset = prepare_split_dataset(
    indices=test_indices,
    original_dataset=pyg_dataset_loaded,
    feature_mean=node_feature_mean,
    feature_std=node_feature_std,
    output_mean=target_mean,
    output_std=target_std,
)


# ------------------------------------------------------------
# 8. DataLoaders
# ------------------------------------------------------------

training_generator = torch.Generator()
training_generator.manual_seed(
    RUN_SEED
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=training_generator,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)


# ------------------------------------------------------------
# 9. GIN regression model
# ------------------------------------------------------------

def create_gin_mlp(
    input_dimension,
    hidden_dimension,
):
    return nn.Sequential(
        nn.Linear(
            input_dimension,
            hidden_dimension,
        ),
        nn.ReLU(),
        nn.Linear(
            hidden_dimension,
            hidden_dimension,
        ),
    )


class GINRegressor(nn.Module):

    def __init__(
        self,
        input_dimension,
        hidden_dimension,
        output_dimension,
        dropout_rate,
    ):
        super().__init__()

        self.gin_1 = GINConv(
            create_gin_mlp(
                input_dimension,
                hidden_dimension,
            ),
            train_eps=True,
        )

        self.batch_norm_1 = nn.BatchNorm1d(
            hidden_dimension
        )

        self.gin_2 = GINConv(
            create_gin_mlp(
                hidden_dimension,
                hidden_dimension,
            ),
            train_eps=True,
        )

        self.batch_norm_2 = nn.BatchNorm1d(
            hidden_dimension
        )

        self.gin_3 = GINConv(
            create_gin_mlp(
                hidden_dimension,
                hidden_dimension,
            ),
            train_eps=True,
        )

        self.batch_norm_3 = nn.BatchNorm1d(
            hidden_dimension
        )

        self.regressor = nn.Sequential(
            nn.Linear(
                hidden_dimension,
                hidden_dimension,
            ),
            nn.ReLU(),
            nn.Dropout(
                dropout_rate
            ),
            nn.Linear(
                hidden_dimension,
                output_dimension,
            ),
        )

    def forward(
        self,
        x,
        edge_index,
        batch,
    ):
        x = self.gin_1(
            x,
            edge_index,
        )

        x = self.batch_norm_1(
            x
        )

        x = torch.relu(
            x
        )

        x = self.gin_2(
            x,
            edge_index,
        )

        x = self.batch_norm_2(
            x
        )

        x = torch.relu(
            x
        )

        x = self.gin_3(
            x,
            edge_index,
        )

        x = self.batch_norm_3(
            x
        )

        x = torch.relu(
            x
        )

        graph_embedding = (
            global_mean_pool(
                x,
                batch,
            )
        )

        return self.regressor(
            graph_embedding
        )


# ------------------------------------------------------------
# 10. Device and optimization
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = GINRegressor(
    input_dimension=5,
    hidden_dimension=HIDDEN_DIMENSION,
    output_dimension=6,
    dropout_rate=DROPOUT_RATE,
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

loss_function = nn.MSELoss()


# ------------------------------------------------------------
# 11. Training and validation functions
# ------------------------------------------------------------

def train_one_epoch(
    model,
    data_loader,
    optimizer,
    loss_function,
    device,
):
    model.train()

    total_loss = 0.0
    total_graphs = 0

    for batch in data_loader:

        batch = batch.to(
            device
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        predictions = model(
            batch.x,
            batch.edge_index,
            batch.batch,
        )

        targets = batch.y.view(
            -1,
            6,
        )

        loss = loss_function(
            predictions,
            targets,
        )

        loss.backward()

        optimizer.step()

        batch_graph_count = (
            batch.num_graphs
        )

        total_loss += (
            loss.item()
            * batch_graph_count
        )

        total_graphs += (
            batch_graph_count
        )

    return (
        total_loss
        / total_graphs
    )


@torch.no_grad()
def evaluate_loss(
    model,
    data_loader,
    loss_function,
    device,
):
    model.eval()

    total_loss = 0.0
    total_graphs = 0

    for batch in data_loader:

        batch = batch.to(
            device
        )

        predictions = model(
            batch.x,
            batch.edge_index,
            batch.batch,
        )

        targets = batch.y.view(
            -1,
            6,
        )

        loss = loss_function(
            predictions,
            targets,
        )

        total_loss += (
            loss.item()
            * batch.num_graphs
        )

        total_graphs += (
            batch.num_graphs
        )

    return (
        total_loss
        / total_graphs
    )


# ------------------------------------------------------------
# 12. Train with early stopping
# ------------------------------------------------------------

best_validation_loss = float(
    "inf"
)

best_epoch = 0
epochs_without_improvement = 0
training_history = []

training_start_time = time.perf_counter()

print("=" * 84)
print("SINGLE-SEED GIN VERIFICATION")
print("=" * 84)

print(
    "Device:",
    device,
)

print(
    "Train/validation/test:",
    len(train_dataset),
    len(validation_dataset),
    len(test_dataset),
)

for epoch in range(
    1,
    MAX_EPOCHS + 1,
):

    training_loss = train_one_epoch(
        model=model,
        data_loader=train_loader,
        optimizer=optimizer,
        loss_function=loss_function,
        device=device,
    )

    validation_loss = evaluate_loss(
        model=model,
        data_loader=validation_loader,
        loss_function=loss_function,
        device=device,
    )

    training_history.append({
        "epoch": epoch,
        "training_loss": (
            training_loss
        ),
        "validation_loss": (
            validation_loss
        ),
    })

    if validation_loss < (
        best_validation_loss
        - 1.0e-7
    ):

        best_validation_loss = (
            validation_loss
        )

        best_epoch = epoch

        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": (
                    model.state_dict()
                ),
                "node_feature_mean": (
                    node_feature_mean
                ),
                "node_feature_std": (
                    node_feature_std
                ),
                "target_mean": (
                    target_mean
                ),
                "target_std": (
                    target_std
                ),
                "seed": RUN_SEED,
                "best_epoch": (
                    best_epoch
                ),
                "best_validation_loss": (
                    best_validation_loss
                ),
            },
            SINGLE_RUN_MODEL_PATH,
        )

    else:

        epochs_without_improvement += 1

    if (
        epoch == 1
        or epoch % 25 == 0
    ):

        print(
            f"Epoch {epoch:03d} | "
            f"Train={training_loss:.6f} | "
            f"Validation={validation_loss:.6f}",
            flush=True,
        )

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):

        print(
            f"Early stopping at epoch "
            f"{epoch}; best epoch="
            f"{best_epoch}",
            flush=True,
        )

        break


training_elapsed_seconds = (
    time.perf_counter()
    - training_start_time
)

pd.DataFrame(
    training_history
).to_csv(
    SINGLE_RUN_HISTORY_PATH,
    index=False,
)


# ------------------------------------------------------------
# 13. Restore best model
# ------------------------------------------------------------

try:
    best_checkpoint = torch.load(
        SINGLE_RUN_MODEL_PATH,
        map_location=device,
        weights_only=False,
    )
except TypeError:
    best_checkpoint = torch.load(
        SINGLE_RUN_MODEL_PATH,
        map_location=device,
    )

model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)


# ------------------------------------------------------------
# 14. Predict on test set
# ------------------------------------------------------------

@torch.no_grad()
def predict_standardized(
    model,
    data_loader,
    device,
):
    model.eval()

    prediction_batches = []
    target_batches = []

    for batch in data_loader:

        batch = batch.to(
            device
        )

        predictions = model(
            batch.x,
            batch.edge_index,
            batch.batch,
        )

        targets = batch.y.view(
            -1,
            6,
        )

        prediction_batches.append(
            predictions.cpu()
        )

        target_batches.append(
            targets.cpu()
        )

    return (
        torch.cat(
            prediction_batches,
            dim=0,
        ),
        torch.cat(
            target_batches,
            dim=0,
        ),
    )


standardized_predictions, standardized_targets = (
    predict_standardized(
        model=model,
        data_loader=test_loader,
        device=device,
    )
)


# ------------------------------------------------------------
# 15. Inverse-transform to original target scale
# ------------------------------------------------------------

predicted_transformed_targets = (
    standardized_predictions
    * target_std
    + target_mean
)

true_transformed_targets = (
    standardized_targets
    * target_std
    + target_mean
)

predicted_original = inverse_targets_from_model(
    predicted_transformed_targets.numpy()
)

true_original = inverse_targets_from_model(
    true_transformed_targets.numpy()
)


# ------------------------------------------------------------
# 16. Calculate original-scale test metrics
# ------------------------------------------------------------

metric_rows = []

for target_index, target_name in enumerate(
    TARGET_COLUMNS
):

    y_true = true_original[
        :,
        target_index,
    ]

    y_pred = predicted_original[
        :,
        target_index,
    ]

    mae = mean_absolute_error(
        y_true,
        y_pred,
    )

    rmse = math.sqrt(
        mean_squared_error(
            y_true,
            y_pred,
        )
    )

    r_squared = r2_score(
        y_true,
        y_pred,
    )

    mape = (
        np.mean(
            np.abs(
                (
                    y_true
                    - y_pred
                )
                / np.maximum(
                    np.abs(y_true),
                    1.0e-8,
                )
            )
        )
        * 100.0
    )

    metric_rows.append({
        "seed": RUN_SEED,
        "target": target_name,
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(
            r_squared
        ),
        "MAPE_percent": float(
            mape
        ),
    })


df_single_run_metrics = pd.DataFrame(
    metric_rows
)

df_single_run_metrics.to_csv(
    SINGLE_RUN_METRICS_PATH,
    index=False,
)


# ------------------------------------------------------------
# 17. Report
# ------------------------------------------------------------

split_summary = (
    df_metadata_loaded.assign(
        split="unused"
    )
)

split_summary.loc[
    train_indices,
    "split",
] = "train"

split_summary.loc[
    validation_indices,
    "split",
] = "validation"

split_summary.loc[
    test_indices,
    "split",
] = "test"

split_composition = (
    split_summary
    .groupby(
        [
            "split",
            "generator_count",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

print("\n" + "=" * 84)
print("SEED-42 TEST RESULTS")
print("=" * 84)

print(
    "Best epoch:",
    best_epoch,
)

print(
    "Best validation loss:",
    f"{best_validation_loss:.8f}",
)

print(
    "Training time:",
    f"{training_elapsed_seconds:.2f} seconds",
)

print("\nSplit composition by k:")
print(
    split_composition.to_string()
)

print("\nOriginal-scale test metrics:")
print(
    df_single_run_metrics.to_string(
        index=False
    )
)

print(
    "\nMetrics saved to:"
)
print(
    SINGLE_RUN_METRICS_PATH
)

print(
    "Training history saved to:"
)
print(
    SINGLE_RUN_HISTORY_PATH
)

print(
    "Best model saved to:"
)
print(
    SINGLE_RUN_MODEL_PATH
)

print_memory_status(
    "RAM status after seed-42 GIN run"
)

del training_node_matrix
del training_transformed_targets
del standardized_predictions
del standardized_targets
del predicted_transformed_targets
del true_transformed_targets
del predicted_original
del true_original
del model
del optimizer

if torch.cuda.is_available():
    torch.cuda.empty_cache()

gc.collect()

print(
    "\nStep 10 completed successfully."
)


SINGLE-SEED GIN VERIFICATION
Device: cuda
Train/validation/test: 140 30 30
Epoch 001 | Train=0.877468 | Validation=0.676032
Epoch 025 | Train=0.056426 | Validation=0.014344
Epoch 050 | Train=0.041766 | Validation=0.012201
Epoch 075 | Train=0.045313 | Validation=0.010554
Epoch 100 | Train=0.030685 | Validation=0.008464
Epoch 125 | Train=0.027029 | Validation=0.007971
Epoch 150 | Train=0.023391 | Validation=0.007126
Epoch 175 | Train=0.024478 | Validation=0.005847
Early stopping at epoch 189; best epoch=129

SEED-42 TEST RESULTS
Best epoch: 129
Best validation loss: 0.00326284
Training time: 13.11 seconds

Split composition by k:
generator_count   2   3   4
split                      
test              9  11  10
train            42  49  49
validation        9  10  11

Original-scale test metrics:
 seed target          MAE         RMSE       R2  MAPE_percent
   42      W 1.022145e+01 1.203076e+01 0.989615      3.277772
   42     MS 1.342284e+02 1.886548e+02 0.994513     10.083275
   42   

In [ ]:
# ============================================================
# STEP 11: Ten repeated stratified GIN experiments
# ============================================================

import gc
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from scipy.stats import t
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from torch_geometric.loader import DataLoader


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

REPEATED_SPLIT_SEEDS = list(
    range(42, 52)
)

BATCH_SIZE = 16
LEARNING_RATE = 1.0e-3
WEIGHT_DECAY = 1.0e-5
MAX_EPOCHS = 500
EARLY_STOPPING_PATIENCE = 60

HIDDEN_DIMENSION = 64
DROPOUT_RATE = 0.10

TARGET_COLUMNS = [
    "W",
    "MS",
    "Z",
    "M1",
    "M2",
    "R",
]

REPEATED_RUN_METRICS_PATH = (
    TABLES_DIR
    / "circulant_repeated_split_run_metrics.csv"
)

REPEATED_SUMMARY_PATH = (
    TABLES_DIR
    / "circulant_repeated_split_summary.csv"
)

REPEATED_TRAINING_SUMMARY_PATH = (
    TABLES_DIR
    / "circulant_repeated_split_training_summary.csv"
)


# ------------------------------------------------------------
# 2. Load compact dataset once
# ------------------------------------------------------------

try:
    pyg_dataset_repeated = torch.load(
        PYG_DATASET_PATH,
        weights_only=False,
    )
except TypeError:
    pyg_dataset_repeated = torch.load(
        PYG_DATASET_PATH
    )

df_metadata_repeated = pd.read_csv(
    PYG_METADATA_PATH,
    dtype={
        "dataset_index": "int32",
        "graph_id": "int32",
        "order": "int32",
        "generator_count": "int8",
        "order_group": "string",
    },
)

assert len(pyg_dataset_repeated) == 200
assert len(df_metadata_repeated) == 200

all_indices = np.arange(
    len(pyg_dataset_repeated)
)

stratification_labels = (
    df_metadata_repeated[
        "generator_count"
    ]
    .astype(str)
    .to_numpy()
)


# ------------------------------------------------------------
# 3. Evaluation helper
# ------------------------------------------------------------

@torch.no_grad()
def predict_on_loader(
    model,
    data_loader,
    device,
):
    model.eval()

    prediction_batches = []
    target_batches = []

    for batch in data_loader:

        batch = batch.to(
            device
        )

        predictions = model(
            batch.x,
            batch.edge_index,
            batch.batch,
        )

        targets = batch.y.view(
            -1,
            6,
        )

        prediction_batches.append(
            predictions.cpu()
        )

        target_batches.append(
            targets.cpu()
        )

    return (
        torch.cat(
            prediction_batches,
            dim=0,
        ),
        torch.cat(
            target_batches,
            dim=0,
        ),
    )


# ------------------------------------------------------------
# 4. Run one repeated split
# ------------------------------------------------------------

def run_repeated_split(seed):

    set_all_random_seeds(
        seed
    )

    # --------------------------------------------------------
    # Split
    # --------------------------------------------------------

    train_indices, temporary_indices = (
        train_test_split(
            all_indices,
            test_size=0.30,
            random_state=seed,
            shuffle=True,
            stratify=stratification_labels,
        )
    )

    temporary_stratification = (
        stratification_labels[
            temporary_indices
        ]
    )

    validation_indices, test_indices = (
        train_test_split(
            temporary_indices,
            test_size=0.50,
            random_state=seed,
            shuffle=True,
            stratify=temporary_stratification,
        )
    )

    assert len(train_indices) == 140
    assert len(validation_indices) == 30
    assert len(test_indices) == 30

    # --------------------------------------------------------
    # Training-only node-feature scaling
    # --------------------------------------------------------

    training_node_matrix = torch.cat(
        [
            pyg_dataset_repeated[
                int(index)
            ].x
            for index in train_indices
        ],
        dim=0,
    )

    feature_mean = training_node_matrix.mean(
        dim=0
    )

    feature_std = training_node_matrix.std(
        dim=0,
        unbiased=False,
    )

    feature_std = torch.where(
        feature_std < 1.0e-8,
        torch.ones_like(
            feature_std
        ),
        feature_std,
    )

    # --------------------------------------------------------
    # Training-only target scaling
    # --------------------------------------------------------

    training_transformed_targets = torch.cat(
        [
            pyg_dataset_repeated[
                int(index)
            ].y
            for index in train_indices
        ],
        dim=0,
    )

    target_mean = training_transformed_targets.mean(
        dim=0
    )

    target_std = training_transformed_targets.std(
        dim=0,
        unbiased=False,
    )

    target_std = torch.where(
        target_std < 1.0e-8,
        torch.ones_like(
            target_std
        ),
        target_std,
    )

    # --------------------------------------------------------
    # Prepare standardized split copies
    # --------------------------------------------------------

    train_dataset = prepare_split_dataset(
        indices=train_indices,
        original_dataset=(
            pyg_dataset_repeated
        ),
        feature_mean=feature_mean,
        feature_std=feature_std,
        output_mean=target_mean,
        output_std=target_std,
    )

    validation_dataset = prepare_split_dataset(
        indices=validation_indices,
        original_dataset=(
            pyg_dataset_repeated
        ),
        feature_mean=feature_mean,
        feature_std=feature_std,
        output_mean=target_mean,
        output_std=target_std,
    )

    test_dataset = prepare_split_dataset(
        indices=test_indices,
        original_dataset=(
            pyg_dataset_repeated
        ),
        feature_mean=feature_mean,
        feature_std=feature_std,
        output_mean=target_mean,
        output_std=target_std,
    )

    training_generator = (
        torch.Generator()
    )

    training_generator.manual_seed(
        seed
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=training_generator,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    model = GINRegressor(
        input_dimension=5,
        hidden_dimension=HIDDEN_DIMENSION,
        output_dimension=6,
        dropout_rate=DROPOUT_RATE,
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    loss_function = nn.MSELoss()

    best_validation_loss = float(
        "inf"
    )

    best_state_dict = None
    best_epoch = 0
    epochs_without_improvement = 0

    start_time = time.perf_counter()

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):

        training_loss = train_one_epoch(
            model=model,
            data_loader=train_loader,
            optimizer=optimizer,
            loss_function=loss_function,
            device=device,
        )

        validation_loss = evaluate_loss(
            model=model,
            data_loader=validation_loader,
            loss_function=loss_function,
            device=device,
        )

        if validation_loss < (
            best_validation_loss
            - 1.0e-7
        ):

            best_validation_loss = (
                validation_loss
            )

            best_epoch = epoch

            epochs_without_improvement = 0

            best_state_dict = {
                key: value.detach().cpu().clone()
                for key, value
                in model.state_dict().items()
            }

        else:

            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):

            break

    training_seconds = (
        time.perf_counter()
        - start_time
    )

    if best_state_dict is None:
        raise RuntimeError(
            f"No valid checkpoint for seed {seed}"
        )

    model.load_state_dict(
        best_state_dict
    )

    model = model.to(
        device
    )

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    standardized_predictions, standardized_targets = (
        predict_on_loader(
            model=model,
            data_loader=test_loader,
            device=device,
        )
    )

    predicted_transformed_targets = (
        standardized_predictions
        * target_std
        + target_mean
    )

    true_transformed_targets = (
        standardized_targets
        * target_std
        + target_mean
    )

    predicted_original = inverse_targets_from_model(
        predicted_transformed_targets.numpy()
    )

    true_original = inverse_targets_from_model(
        true_transformed_targets.numpy()
    )

    # --------------------------------------------------------
    # Original-scale metrics
    # --------------------------------------------------------

    metric_rows = []

    for target_index, target_name in enumerate(
        TARGET_COLUMNS
    ):

        y_true = true_original[
            :,
            target_index,
        ]

        y_pred = predicted_original[
            :,
            target_index,
        ]

        mae = mean_absolute_error(
            y_true,
            y_pred,
        )

        rmse = math.sqrt(
            mean_squared_error(
                y_true,
                y_pred,
            )
        )

        r_squared = r2_score(
            y_true,
            y_pred,
        )

        mape = (
            np.mean(
                np.abs(
                    (
                        y_true
                        - y_pred
                    )
                    /
                    np.maximum(
                        np.abs(y_true),
                        1.0e-8,
                    )
                )
            )
            * 100.0
        )

        metric_rows.append({
            "seed": seed,
            "target": target_name,
            "MAE": float(mae),
            "RMSE": float(rmse),
            "R2": float(r_squared),
            "MAPE_percent": float(mape),
            "best_epoch": int(
                best_epoch
            ),
            "best_validation_loss": float(
                best_validation_loss
            ),
            "training_seconds": float(
                training_seconds
            ),
            "train_graphs": 140,
            "validation_graphs": 30,
            "test_graphs": 30,
        })

    # --------------------------------------------------------
    # Release split-specific memory
    # --------------------------------------------------------

    del training_node_matrix
    del training_transformed_targets
    del train_dataset
    del validation_dataset
    del test_dataset
    del train_loader
    del validation_loader
    del test_loader
    del standardized_predictions
    del standardized_targets
    del predicted_transformed_targets
    del true_transformed_targets
    del predicted_original
    del true_original
    del best_state_dict
    del model
    del optimizer

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()

    return metric_rows


# ------------------------------------------------------------
# 5. Execute all 10 seeds
# ------------------------------------------------------------

all_metric_rows = []

print("=" * 92)
print("TEN REPEATED STRATIFIED GIN RUNS")
print("=" * 92)

for run_number, seed in enumerate(
    REPEATED_SPLIT_SEEDS,
    start=1,
):

    print(
        f"\nRun {run_number}/"
        f"{len(REPEATED_SPLIT_SEEDS)} "
        f"| seed={seed}",
        flush=True,
    )

    run_metric_rows = (
        run_repeated_split(
            seed
        )
    )

    all_metric_rows.extend(
        run_metric_rows
    )

    run_dataframe = pd.DataFrame(
        run_metric_rows
    )

    print(
        run_dataframe[
            [
                "target",
                "R2",
                "MAPE_percent",
            ]
        ].to_string(
            index=False
        ),
        flush=True,
    )

    # Save after every seed for interruption safety
    pd.DataFrame(
        all_metric_rows
    ).to_csv(
        REPEATED_RUN_METRICS_PATH,
        index=False,
    )

    print_memory_status(
        f"RAM after seed {seed}"
    )


# ------------------------------------------------------------
# 6. Aggregate repeated-run results
# ------------------------------------------------------------

df_repeated_metrics = pd.DataFrame(
    all_metric_rows
)

assert len(df_repeated_metrics) == 60
assert (
    df_repeated_metrics["seed"].nunique()
    == 10
)

summary_rows = []

confidence_level = 0.95
degrees_of_freedom = (
    len(REPEATED_SPLIT_SEEDS)
    - 1
)

critical_t = t.ppf(
    0.5 + confidence_level / 2.0,
    degrees_of_freedom,
)

for target_name in TARGET_COLUMNS:

    current = df_repeated_metrics.loc[
        df_repeated_metrics[
            "target"
        ] == target_name
    ]

    summary_record = {
        "target": target_name
    }

    for metric_name in [
        "MAE",
        "RMSE",
        "R2",
        "MAPE_percent",
    ]:

        values = current[
            metric_name
        ].to_numpy(
            dtype=float
        )

        mean_value = float(
            np.mean(values)
        )

        standard_deviation = float(
            np.std(
                values,
                ddof=1,
            )
        )

        standard_error = (
            standard_deviation
            / math.sqrt(
                len(values)
            )
        )

        confidence_half_width = (
            critical_t
            * standard_error
        )

        summary_record[
            f"{metric_name}_mean"
        ] = mean_value

        summary_record[
            f"{metric_name}_std"
        ] = standard_deviation

        summary_record[
            f"{metric_name}_CI95_lower"
        ] = (
            mean_value
            - confidence_half_width
        )

        summary_record[
            f"{metric_name}_CI95_upper"
        ] = (
            mean_value
            + confidence_half_width
        )

    summary_rows.append(
        summary_record
    )


df_repeated_summary = pd.DataFrame(
    summary_rows
)

df_repeated_summary.to_csv(
    REPEATED_SUMMARY_PATH,
    index=False,
)


# ------------------------------------------------------------
# 7. Training summary
# ------------------------------------------------------------

df_training_summary = (
    df_repeated_metrics[
        [
            "seed",
            "best_epoch",
            "best_validation_loss",
            "training_seconds",
        ]
    ]
    .drop_duplicates(
        subset=["seed"]
    )
    .sort_values("seed")
    .reset_index(drop=True)
)

df_training_summary.to_csv(
    REPEATED_TRAINING_SUMMARY_PATH,
    index=False,
)


# ------------------------------------------------------------
# 8. Compact manuscript-ready display
# ------------------------------------------------------------

display_rows = []

for _, row in (
    df_repeated_summary.iterrows()
):

    display_rows.append({
        "Target": row["target"],
        "MAE": (
            f"{row['MAE_mean']:.4g}"
            f" ± "
            f"{row['MAE_std']:.3g}"
        ),
        "RMSE": (
            f"{row['RMSE_mean']:.4g}"
            f" ± "
            f"{row['RMSE_std']:.3g}"
        ),
        "R2": (
            f"{row['R2_mean']:.4f}"
            f" ± "
            f"{row['R2_std']:.4f}"
        ),
        "R2_95_CI": (
            f"["
            f"{row['R2_CI95_lower']:.4f}, "
            f"{row['R2_CI95_upper']:.4f}"
            f"]"
        ),
        "MAPE_percent": (
            f"{row['MAPE_percent_mean']:.2f}"
            f" ± "
            f"{row['MAPE_percent_std']:.2f}"
        ),
    })


df_repeated_display = pd.DataFrame(
    display_rows
)


# ------------------------------------------------------------
# 9. Report
# ------------------------------------------------------------

print("\n" + "=" * 92)
print("REPEATED-SPLIT SUMMARY")
print("=" * 92)

print(
    df_repeated_display.to_string(
        index=False
    )
)

print("\nTraining summary:")
print(
    df_training_summary.to_string(
        index=False
    )
)

print(
    "\nRun-level metrics saved to:"
)
print(
    REPEATED_RUN_METRICS_PATH
)

print(
    "Repeated-split summary saved to:"
)
print(
    REPEATED_SUMMARY_PATH
)

print(
    "Training summary saved to:"
)
print(
    REPEATED_TRAINING_SUMMARY_PATH
)

print_memory_status(
    "RAM after all repeated runs"
)

gc.collect()

print(
    "\nStep 11 completed successfully."
)


TEN REPEATED STRATIFIED GIN RUNS

Run 1/10 | seed=42
target       R2  MAPE_percent
     W 0.989615      3.277772
    MS 0.994513     10.083275
     Z 0.978393     13.861769
    M1 0.998244      2.049337
    M2 0.998024      3.550777
     R 0.994329      1.006705

RAM after seed 42
----------------------------------------------------------------------
process_ram_gb: 1.498
system_ram_total_gb: 12.671
system_ram_available_gb: 10.148
system_ram_used_percent: 19.900

Run 2/10 | seed=43
target       R2  MAPE_percent
     W 0.995818      1.957388
    MS 0.999154      4.672655
     Z 0.975983     13.139151
    M1 0.998764      1.783065
    M2 0.998323      2.961425
     R 0.995673      0.847594

RAM after seed 43
----------------------------------------------------------------------
process_ram_gb: 1.498
system_ram_total_gb: 12.671
system_ram_available_gb: 10.181
system_ram_used_percent: 19.700

Run 3/10 | seed=44
target       R2  MAPE_percent
     W 0.990363      3.158040
    MS 0.979086    

In [ ]:
# ============================================================
# STEP 12: Order-held-out GIN evaluation
# Train/validation orders: 16--23
# Test orders: 24--26
# ============================================================

import gc
import math
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from torch_geometric.loader import DataLoader


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

ORDER_HOLDOUT_SEED = 42

BATCH_SIZE = 16
LEARNING_RATE = 1.0e-3
WEIGHT_DECAY = 1.0e-5
MAX_EPOCHS = 500
EARLY_STOPPING_PATIENCE = 60

HIDDEN_DIMENSION = 64
DROPOUT_RATE = 0.10

TARGET_COLUMNS = [
    "W",
    "MS",
    "Z",
    "M1",
    "M2",
    "R",
]

ORDER_HOLDOUT_METRICS_PATH = (
    TABLES_DIR
    / "circulant_order_holdout_metrics.csv"
)

ORDER_HOLDOUT_COMPOSITION_PATH = (
    TABLES_DIR
    / "circulant_order_holdout_composition.csv"
)

ORDER_HOLDOUT_HISTORY_PATH = (
    RESULTS_DIR
    / "logs"
    / "circulant_order_holdout_training_history.csv"
)

ORDER_HOLDOUT_MODEL_PATH = (
    MODELS_DIR
    / "circulant_order_holdout_best_gin.pt"
)


# ------------------------------------------------------------
# 2. Reproducibility
# ------------------------------------------------------------

set_all_random_seeds(
    ORDER_HOLDOUT_SEED
)


# ------------------------------------------------------------
# 3. Load dataset and metadata
# ------------------------------------------------------------

try:
    pyg_dataset_holdout = torch.load(
        PYG_DATASET_PATH,
        weights_only=False,
    )
except TypeError:
    pyg_dataset_holdout = torch.load(
        PYG_DATASET_PATH
    )

df_metadata_holdout = pd.read_csv(
    PYG_METADATA_PATH,
    dtype={
        "dataset_index": "int32",
        "graph_id": "int32",
        "order": "int32",
        "generator_count": "int8",
        "order_group": "string",
    },
)

assert len(pyg_dataset_holdout) == 200
assert len(df_metadata_holdout) == 200


# ------------------------------------------------------------
# 4. Define development and held-out indices
# ------------------------------------------------------------

development_indices = (
    df_metadata_holdout.loc[
        df_metadata_holdout[
            "order_group"
        ] == "development",
        "dataset_index",
    ]
    .to_numpy(
        dtype=int
    )
)

heldout_test_indices = (
    df_metadata_holdout.loc[
        df_metadata_holdout[
            "order_group"
        ] == "heldout",
        "dataset_index",
    ]
    .to_numpy(
        dtype=int
    )
)

assert len(development_indices) == 155
assert len(heldout_test_indices) == 45

assert set(
    development_indices
).isdisjoint(
    heldout_test_indices
)


# ------------------------------------------------------------
# 5. Split development graphs into train and validation
# ------------------------------------------------------------

development_stratification = (
    df_metadata_holdout.loc[
        development_indices,
        "generator_count",
    ]
    .astype(str)
    .to_numpy()
)

train_indices, validation_indices = (
    train_test_split(
        development_indices,
        test_size=0.20,
        random_state=ORDER_HOLDOUT_SEED,
        shuffle=True,
        stratify=development_stratification,
    )
)

assert len(train_indices) == 124
assert len(validation_indices) == 31
assert len(heldout_test_indices) == 45


# ------------------------------------------------------------
# 6. Training-only node-feature scaling
# ------------------------------------------------------------

training_node_matrix = torch.cat(
    [
        pyg_dataset_holdout[
            int(index)
        ].x
        for index in train_indices
    ],
    dim=0,
)

node_feature_mean = (
    training_node_matrix.mean(
        dim=0
    )
)

node_feature_std = (
    training_node_matrix.std(
        dim=0,
        unbiased=False,
    )
)

node_feature_std = torch.where(
    node_feature_std < 1.0e-8,
    torch.ones_like(
        node_feature_std
    ),
    node_feature_std,
)


# ------------------------------------------------------------
# 7. Training-only target scaling
# ------------------------------------------------------------

training_transformed_targets = torch.cat(
    [
        pyg_dataset_holdout[
            int(index)
        ].y
        for index in train_indices
    ],
    dim=0,
)

target_mean = (
    training_transformed_targets.mean(
        dim=0
    )
)

target_std = (
    training_transformed_targets.std(
        dim=0,
        unbiased=False,
    )
)

target_std = torch.where(
    target_std < 1.0e-8,
    torch.ones_like(
        target_std
    ),
    target_std,
)


# ------------------------------------------------------------
# 8. Prepare standardized datasets
# ------------------------------------------------------------

train_dataset = prepare_split_dataset(
    indices=train_indices,
    original_dataset=pyg_dataset_holdout,
    feature_mean=node_feature_mean,
    feature_std=node_feature_std,
    output_mean=target_mean,
    output_std=target_std,
)

validation_dataset = prepare_split_dataset(
    indices=validation_indices,
    original_dataset=pyg_dataset_holdout,
    feature_mean=node_feature_mean,
    feature_std=node_feature_std,
    output_mean=target_mean,
    output_std=target_std,
)

heldout_test_dataset = prepare_split_dataset(
    indices=heldout_test_indices,
    original_dataset=pyg_dataset_holdout,
    feature_mean=node_feature_mean,
    feature_std=node_feature_std,
    output_mean=target_mean,
    output_std=target_std,
)


# ------------------------------------------------------------
# 9. DataLoaders
# ------------------------------------------------------------

training_generator = torch.Generator()
training_generator.manual_seed(
    ORDER_HOLDOUT_SEED
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=training_generator,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

heldout_test_loader = DataLoader(
    heldout_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)


# ------------------------------------------------------------
# 10. Model
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = GINRegressor(
    input_dimension=5,
    hidden_dimension=HIDDEN_DIMENSION,
    output_dimension=6,
    dropout_rate=DROPOUT_RATE,
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

loss_function = nn.MSELoss()


# ------------------------------------------------------------
# 11. Training with early stopping
# ------------------------------------------------------------

best_validation_loss = float(
    "inf"
)

best_epoch = 0
epochs_without_improvement = 0
training_history = []

training_start_time = time.perf_counter()

print("=" * 88)
print("ORDER-HELD-OUT GIN EVALUATION")
print("=" * 88)

print(
    "Device:",
    device,
)

print(
    "Train/validation/test:",
    len(train_dataset),
    len(validation_dataset),
    len(heldout_test_dataset),
)

print(
    "Training/validation orders:",
    sorted(
        df_metadata_holdout.loc[
            np.concatenate(
                [
                    train_indices,
                    validation_indices,
                ]
            ),
            "order",
        ].unique()
    ),
)

print(
    "Held-out test orders:",
    sorted(
        df_metadata_holdout.loc[
            heldout_test_indices,
            "order",
        ].unique()
    ),
)

for epoch in range(
    1,
    MAX_EPOCHS + 1,
):

    training_loss = train_one_epoch(
        model=model,
        data_loader=train_loader,
        optimizer=optimizer,
        loss_function=loss_function,
        device=device,
    )

    validation_loss = evaluate_loss(
        model=model,
        data_loader=validation_loader,
        loss_function=loss_function,
        device=device,
    )

    training_history.append({
        "epoch": epoch,
        "training_loss": training_loss,
        "validation_loss": validation_loss,
    })

    if validation_loss < (
        best_validation_loss
        - 1.0e-7
    ):

        best_validation_loss = (
            validation_loss
        )

        best_epoch = epoch

        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": (
                    model.state_dict()
                ),
                "node_feature_mean": (
                    node_feature_mean
                ),
                "node_feature_std": (
                    node_feature_std
                ),
                "target_mean": target_mean,
                "target_std": target_std,
                "seed": ORDER_HOLDOUT_SEED,
                "best_epoch": best_epoch,
                "best_validation_loss": (
                    best_validation_loss
                ),
                "training_orders": list(
                    range(16, 24)
                ),
                "heldout_orders": [
                    24,
                    25,
                    26,
                ],
            },
            ORDER_HOLDOUT_MODEL_PATH,
        )

    else:

        epochs_without_improvement += 1

    if (
        epoch == 1
        or epoch % 25 == 0
    ):

        print(
            f"Epoch {epoch:03d} | "
            f"Train={training_loss:.6f} | "
            f"Validation={validation_loss:.6f}",
            flush=True,
        )

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):

        print(
            f"Early stopping at epoch "
            f"{epoch}; best epoch="
            f"{best_epoch}",
            flush=True,
        )

        break


training_elapsed_seconds = (
    time.perf_counter()
    - training_start_time
)

pd.DataFrame(
    training_history
).to_csv(
    ORDER_HOLDOUT_HISTORY_PATH,
    index=False,
)


# ------------------------------------------------------------
# 12. Restore best checkpoint
# ------------------------------------------------------------

try:
    best_checkpoint = torch.load(
        ORDER_HOLDOUT_MODEL_PATH,
        map_location=device,
        weights_only=False,
    )
except TypeError:
    best_checkpoint = torch.load(
        ORDER_HOLDOUT_MODEL_PATH,
        map_location=device,
    )

model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)


# ------------------------------------------------------------
# 13. Held-out predictions
# ------------------------------------------------------------

standardized_predictions, standardized_targets = (
    predict_standardized(
        model=model,
        data_loader=heldout_test_loader,
        device=device,
    )
)

predicted_transformed_targets = (
    standardized_predictions
    * target_std
    + target_mean
)

true_transformed_targets = (
    standardized_targets
    * target_std
    + target_mean
)

predicted_original = inverse_targets_from_model(
    predicted_transformed_targets.numpy()
)

true_original = inverse_targets_from_model(
    true_transformed_targets.numpy()
)


# ------------------------------------------------------------
# 14. Original-scale metrics
# ------------------------------------------------------------

metric_rows = []

for target_index, target_name in enumerate(
    TARGET_COLUMNS
):

    y_true = true_original[
        :,
        target_index,
    ]

    y_pred = predicted_original[
        :,
        target_index,
    ]

    mae = mean_absolute_error(
        y_true,
        y_pred,
    )

    rmse = math.sqrt(
        mean_squared_error(
            y_true,
            y_pred,
        )
    )

    r_squared = r2_score(
        y_true,
        y_pred,
    )

    mape = (
        np.mean(
            np.abs(
                (
                    y_true
                    - y_pred
                )
                /
                np.maximum(
                    np.abs(y_true),
                    1.0e-8,
                )
            )
        )
        * 100.0
    )

    metric_rows.append({
        "seed": ORDER_HOLDOUT_SEED,
        "target": target_name,
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r_squared),
        "MAPE_percent": float(mape),
        "train_graphs": int(
            len(train_dataset)
        ),
        "validation_graphs": int(
            len(validation_dataset)
        ),
        "test_graphs": int(
            len(heldout_test_dataset)
        ),
        "best_epoch": int(
            best_epoch
        ),
        "best_validation_loss": float(
            best_validation_loss
        ),
        "training_seconds": float(
            training_elapsed_seconds
        ),
    })


df_order_holdout_metrics = pd.DataFrame(
    metric_rows
)

df_order_holdout_metrics.to_csv(
    ORDER_HOLDOUT_METRICS_PATH,
    index=False,
)


# ------------------------------------------------------------
# 15. Composition table
# ------------------------------------------------------------

composition_records = []

split_definitions = {
    "Training": train_indices,
    "Validation": validation_indices,
    "Test": heldout_test_indices,
}

for split_name, split_indices in (
    split_definitions.items()
):

    current_metadata = (
        df_metadata_holdout.loc[
            split_indices
        ]
    )

    composition_records.append({
        "subset": split_name,
        "minimum_order": int(
            current_metadata[
                "order"
            ].min()
        ),
        "maximum_order": int(
            current_metadata[
                "order"
            ].max()
        ),
        "graphs": int(
            len(current_metadata)
        ),
        "k2_graphs": int(
            (
                current_metadata[
                    "generator_count"
                ] == 2
            ).sum()
        ),
        "k3_graphs": int(
            (
                current_metadata[
                    "generator_count"
                ] == 3
            ).sum()
        ),
        "k4_graphs": int(
            (
                current_metadata[
                    "generator_count"
                ] == 4
            ).sum()
        ),
    })


df_order_holdout_composition = pd.DataFrame(
    composition_records
)

df_order_holdout_composition.to_csv(
    ORDER_HOLDOUT_COMPOSITION_PATH,
    index=False,
)


# ------------------------------------------------------------
# 16. Report
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("ORDER-HELD-OUT RESULTS")
print("=" * 88)

print(
    "Best epoch:",
    best_epoch,
)

print(
    "Best validation loss:",
    f"{best_validation_loss:.8f}",
)

print(
    "Training time:",
    f"{training_elapsed_seconds:.2f} seconds",
)

print("\nSplit composition:")
print(
    df_order_holdout_composition.to_string(
        index=False
    )
)

print("\nOriginal-scale held-out metrics:")
print(
    df_order_holdout_metrics[
        [
            "target",
            "MAE",
            "RMSE",
            "R2",
            "MAPE_percent",
        ]
    ].to_string(
        index=False
    )
)

print(
    "\nMetrics saved to:"
)
print(
    ORDER_HOLDOUT_METRICS_PATH
)

print(
    "Composition saved to:"
)
print(
    ORDER_HOLDOUT_COMPOSITION_PATH
)

print(
    "Training history saved to:"
)
print(
    ORDER_HOLDOUT_HISTORY_PATH
)

print(
    "Best model saved to:"
)
print(
    ORDER_HOLDOUT_MODEL_PATH
)

print_memory_status(
    "RAM after order-held-out run"
)

del training_node_matrix
del training_transformed_targets
del standardized_predictions
del standardized_targets
del predicted_transformed_targets
del true_transformed_targets
del predicted_original
del true_original
del model
del optimizer

if torch.cuda.is_available():
    torch.cuda.empty_cache()

gc.collect()

print(
    "\nStep 12 completed successfully."
)


ORDER-HELD-OUT GIN EVALUATION
Device: cuda
Train/validation/test: 124 31 45
Training/validation orders: [np.int32(16), np.int32(17), np.int32(18), np.int32(19), np.int32(20), np.int32(21), np.int32(22), np.int32(23)]
Held-out test orders: [np.int32(24), np.int32(25), np.int32(26)]
Epoch 001 | Train=0.929702 | Validation=0.768879
Epoch 025 | Train=0.106718 | Validation=0.020146
Epoch 050 | Train=0.052987 | Validation=0.010335
Epoch 075 | Train=0.040303 | Validation=0.005359
Epoch 100 | Train=0.031299 | Validation=0.013425
Epoch 125 | Train=0.027892 | Validation=0.007962
Epoch 150 | Train=0.022615 | Validation=0.005046
Epoch 175 | Train=0.026666 | Validation=0.005542
Epoch 200 | Train=0.020823 | Validation=0.003714
Early stopping at epoch 224; best epoch=164

ORDER-HELD-OUT RESULTS
Best epoch: 164
Best validation loss: 0.00358375
Training time: 13.07 seconds

Split composition:
    subset  minimum_order  maximum_order  graphs  k2_graphs  k3_graphs  k4_graphs
  Training             16    